# CS781 Phase 3: full-document section tracker

Run All in the **same Kaggle session with both GPUs enabled**. This revision repairs the two-GPU auxiliary-loss crash and the later recovery-ZIP barrier timeout. It detects either failed run's saved configuration and reuses completed checkpoints and feature files. If the Kaggle session has restarted, attach the latest `CS781_Phase3_Sections_All_Artifacts.zip` from the failed run as a Kaggle Input. The live six-run files or `CS781_Phase3_All_Artifacts.zip` can also supply the original inputs. It needs no Hugging Face token or external model. It restores the user's Phase 1 base 03 and official Phase 3 data. The prior submission is read only to audit changed labels; it is never used as the prediction.

**Repair build (2026-10-13):** upload this notebook as a new Kaggle notebook. A saved copy of the earlier execution still has `u.snapshot()` inside the distributed workers and will repeat the `outer_1` NCCL timeout. This build archives only after both workers exit.

This experiment trains a target-aware local encoder with document-balanced sampling, generates out-of-sample features by document cross-fitting, and trains a full-document model with role, boundary, speaker, precedent-stance, and span objectives. A duration-limited section decoder can represent sections longer than 64 sentences by adjoining same-role spans. Five outer document folds evaluate the candidates; the final model is trained on all 80 training documents. Official development labels select checkpoints. Test labels are never read.

The final path runs first, so a **new** submission and recovery archive are created before the outer-fold audit. Recovery ZIP writing now runs only after distributed workers have exited; completed optimizer checkpoints are pruned to keep archives manageable. An interrupted Run All can resume from its own `CS781_Phase3_Sections_All_Artifacts.zip`. The local encoder and feature extraction use two-GPU distributed execution; the full-document head also trains on both GPUs. Model, feature, prediction, and metric files, plus optimizer checkpoints for unfinished stages, are kept for recovery. After fold 0, the notebook stops further folds when every new candidate is below 0.50 supported macro F1 on that held-out fold. This avoids spending hours on a weak pilot. Otherwise expect a run of up to about 9.5 hours; actual time depends on Kaggle hardware.

**Outputs:** `/kaggle/working/CS781_Phase3_Sections_Submission.zip` contains root-level `prediction.json`. `/kaggle/working/CS781_Phase3_Sections_All_Artifacts.zip` contains recovery files. `/kaggle/working/submission.zip` is a convenience copy. Read `phase3_sections/report.json` and `selection.json` before uploading. A private macro F1 of 0.8 is a target, not a guaranteed result; only Kaggle can measure it.

## Embedded recovery and tokenizer code

In [ ]:
%%writefile /kaggle/working/cs781_phase3_hierarchical_runner.py
"""CS781 Phase 3: document-held-out, target-aware hierarchical run.

Embedded verbatim in the Run All notebook. Uses only the user's Phase 1 model
03, official Phase 3 data, and the already trained Phase 3 slot 06 fallback.
"""
from __future__ import annotations

import argparse
import hashlib
import importlib.util
import json
import math
import os
import random
import shutil
import sys
import tempfile
import time
import zipfile
from collections import Counter
from contextlib import nullcontext
from datetime import timedelta
from pathlib import Path

import torch
import torch.distributed as dist
from torch import nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.checkpoint import checkpoint

WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
PRIOR = WORK / 'CS781_Capstone/phase3'
ROOT = WORK / 'CS781_Capstone/phase3_hier'
ARTIFACT = WORK / 'CS781_Phase3_Hierarchical_All_Artifacts.zip'
SUBMISSION = WORK / 'CS781_Phase3_Hierarchical_Submission.zip'
OLD_ARCHIVES = ('CS781_Phase3_All_Artifacts.zip', 'CS781_Phase3_One_All_Artifacts.zip')
FOLDS = 5
SEED = 78123
MAX_TOKENS = 192
BATCH = 32
TOKEN_BUDGET = 6144
MAX_BATCH = 96
ACCUM = 2
SENT_EPOCHS = 10
HEAD_EPOCHS = 20
WINDOW = 128
STRIDE = 96


def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(4 << 20), b''):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', dir=path.parent, encoding='utf8', delete=False) as f:
        temp = Path(f.name)
        json.dump(value, f, indent=2, ensure_ascii=False)
        f.write('\n'); f.flush(); os.fsync(f.fileno())
    os.replace(temp, path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf8'))


def atomic_torch(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as f:
        temp = Path(f.name)
    try:
        torch.save(obj, temp)
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def safe_name(name):
    p = Path(name)
    return bool(name) and not p.is_absolute() and '..' not in p.parts and '\\' not in name


def locate(name):
    direct = WORK / name
    if direct.is_file():
        return direct
    hits = sorted(p for p in INPUT.rglob(name) if p.is_file())
    if len(hits) > 1:
        raise RuntimeError(f'Multiple {name} inputs: {hits}')
    return hits[0] if hits else None


def required_prior(name):
    return (name == 'runner.py' or name.startswith('official/') or
            name.startswith('bases/03/') or name == 'slot_06/best.pt')


def extract_manifest_zip(source, prefix, destination, selector):
    """Selectively extract a manifest archive; verify each selected member."""
    with zipfile.ZipFile(source) as z:
        names = z.namelist()
        if len(set(names)) != len(names) or 'integrity.json' not in names:
            raise RuntimeError(f'Bad recovery member list: {source}')
        manifest = json.loads(z.read('integrity.json'))['files']
        if set(manifest) != set(names) - {'integrity.json'}:
            raise RuntimeError(f'Bad recovery manifest: {source}')
        selected = 0
        for name in names:
            if name == 'integrity.json':
                continue
            if not safe_name(name) or z.getinfo(name).is_dir():
                raise RuntimeError(f'Unsafe recovery path: {name}')
            if not name.startswith(prefix):
                continue
            relative = name[len(prefix):]
            if not selector(relative):
                continue
            dest = destination / relative
            info = manifest[name]
            if dest.is_file() and dest.stat().st_size == info['bytes'] and digest(dest) == info['sha256']:
                selected += 1; continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            with z.open(name) as src, tempfile.NamedTemporaryFile(dir=dest.parent, delete=False) as out:
                temp = Path(out.name); h = hashlib.sha256()
                for block in iter(lambda: src.read(4 << 20), b''):
                    h.update(block); out.write(block)
            if temp.stat().st_size != info['bytes'] or h.hexdigest() != info['sha256']:
                temp.unlink(missing_ok=True)
                raise RuntimeError(f'Corrupt recovery member {name}')
            os.replace(temp, dest); selected += 1
    return selected


def prior_ready():
    needs = ('runner.py', 'official/CL_train.json', 'official/IT_test.json',
             'official/label_vocab.json', 'bases/03/model/prefix_hybrid_lm.py',
             'bases/03/model/course_bpe.py', 'bases/03/checkpoints/selected_model.pt',
             'bases/03/tokenizer/tokenizer.json', 'slot_06/best.pt')
    return all((PRIOR / x).is_file() for x in needs)


def restore():
    own = locate(ARTIFACT.name)
    if own is not None and not (ROOT / 'config.json').is_file():
        extract_manifest_zip(own, 'prior/', PRIOR, required_prior)
        extract_manifest_zip(own, 'hier/', ROOT, lambda _: True)
    if not prior_ready():
        for name in OLD_ARCHIVES:
            source = locate(name)
            if source is None:
                continue
            prefix = '' if name == OLD_ARCHIVES[0] else 'prior/'
            count = extract_manifest_zip(source, prefix, PRIOR, required_prior)
            print('Restored', count, 'prior files from', source, flush=True)
            if prior_ready():
                break
    if not prior_ready():
        raise FileNotFoundError('Need the live six-run Kaggle session or attach CS781_Phase3_All_Artifacts.zip')


def snapshot():
    """Portable archive with model states, optimizer checkpoints, data and code."""
    if not prior_ready():
        raise RuntimeError('Prior run is not staged')
    sources = [(p, 'prior/' + p.relative_to(PRIOR).as_posix())
               for p in PRIOR.rglob('*') if p.is_file() and required_prior(p.relative_to(PRIOR).as_posix())
               and '__pycache__' not in p.parts]
    sources += [(p, 'hier/' + p.relative_to(ROOT).as_posix())
                for p in ROOT.rglob('*') if p.is_file() and not p.name.endswith('.tmp')]
    sources.sort(key=lambda item: item[1])
    manifest = {'version': 2, 'files': {}}
    temp = ARTIFACT.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp, 'w', allowZip64=True) as z:
        for path, name in sources:
            manifest['files'][name] = {'bytes': path.stat().st_size, 'sha256': digest(path)}
            z.write(path, name, compress_type=zipfile.ZIP_STORED if path.suffix in {'.pt', '.zip'}
                    else zipfile.ZIP_DEFLATED)
        z.writestr('integrity.json', json.dumps(manifest, sort_keys=True))
    os.replace(temp, ARTIFACT)
    print('Recovery archive:', ARTIFACT, round(ARTIFACT.stat().st_size / 2**30, 2), 'GiB', flush=True)


def legacy():
    spec = importlib.util.spec_from_file_location('cs781_phase3_legacy', PRIOR / 'runner.py')
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


def sets():
    old = legacy()
    train, dev, test = old.all_data()
    vocab = old.labels()
    if len(train) != 80 or len(dev) != 10 or len(test) != 10:
        raise RuntimeError('Unexpected official Phase 3 document counts')
    return old, train, dev, test, vocab


def fold_map(train):
    rng = random.Random(SEED)
    mapping = {}
    for domain in ('CL', 'IT'):
        rows = [d for d in train if d['domain'] == domain]
        rng.shuffle(rows)
        # Greedy length balance after a seeded shuffle to avoid one giant fold.
        groups = [[] for _ in range(FOLDS)]
        totals = [0] * FOLDS
        for doc in sorted(rows, key=lambda d: len(d['text']), reverse=True):
            candidates = [i for i in range(FOLDS) if len(groups[i]) < 8]
            f = min(candidates, key=lambda i: (totals[i], len(groups[i]), i))
            groups[f].append(doc); totals[f] += len(doc['text'])
        for i, group in enumerate(groups):
            if len(group) != 8:
                raise RuntimeError('Fold must hold out eight documents per domain')
            for doc in group:
                mapping[doc['id']] = i
    return mapping


def target_ids(tokenizer, text):
    seq = tokenizer.backend.encode(text, add_special_tokens=False).ids
    if len(seq) > MAX_TOKENS:
        front = MAX_TOKENS * 3 // 4
        seq = seq[:front] + seq[-(MAX_TOKENS-front):]
    return seq or [int(tokenizer.pad_token_id)]


def prepare():
    if torch.cuda.device_count() != 2:
        raise RuntimeError('Enable two Kaggle GPUs before Run All')
    restore()
    old, train, dev, test, vocab = sets()
    ROOT.mkdir(parents=True, exist_ok=True)
    config = {'folds': FOLDS, 'seed': SEED, 'max_tokens': MAX_TOKENS,
              'batch': BATCH, 'token_budget': TOKEN_BUDGET, 'max_batch': MAX_BATCH,
              'accum': ACCUM, 'window': WINDOW, 'stride': STRIDE,
              'sent_epochs': SENT_EPOCHS, 'head_epochs': HEAD_EPOCHS,
              'prior_source_sha256': digest(PRIOR / 'runner.py'),
              'source_sha256': digest(__file__), 'vocab': vocab,
              'fold_map': fold_map(train), 'train_ids': [d['id'] for d in train],
              'dev_ids': [d['id'] for d in dev], 'test_ids': [d['id'] for d in test]}
    path = ROOT / 'config.json'
    if path.exists() and read_json(path) != config:
        raise RuntimeError('Existing recovery has different data or code; resume with its original notebook')
    atomic_json(path, config)
    shutil.copyfile(__file__, ROOT / 'hierarchical_runner.py')
    cache = ROOT / 'tokens.pt'
    if not cache.exists():
        _, tokenizer = old.load_base('03')
        data = {d['id']: [target_ids(tokenizer, s) for s in d['text']]
                for d in train + dev + test}
        atomic_torch(cache, data)
        del tokenizer
    data = load_tokens()
    lengths = sorted(len(seq) for rows in data.values() for seq in rows)
    atomic_json(ROOT / 'data_audit.json', {
        'documents': {'train': len(train), 'dev': len(dev), 'test': len(test)},
        'sentences': {'train': sum(len(d['text']) for d in train),
                      'dev': sum(len(d['text']) for d in dev),
                      'test': sum(len(d['text']) for d in test)},
        'train_label_counts': dict(Counter(y for d in train for y in d['labels'])),
        'dev_label_counts': dict(Counter(y for d in dev for y in d['labels'])),
        'token_length_p50': lengths[len(lengths)//2],
        'token_length_p99': lengths[int(.99*(len(lengths)-1))],
        'max_target_tokens': MAX_TOKENS,
        'truncated_target_count_at_limit': sum(x == MAX_TOKENS for x in lengths),
        'five_document_held_out_folds': True,
        'fold_training_documents': 60, 'fold_inner_validation_documents': 4,
        'fold_outer_holdout_documents': 16,
        'private_score_reference_slot06': .5058})
    print('Prepared', len(train), 'train documents;', len(dev), 'dev;', len(test), 'test.', flush=True)
    snapshot()


def load_tokens():
    return torch.load(ROOT / 'tokens.pt', map_location='cpu', weights_only=False)


def init_dist():
    rank = int(os.environ.get('RANK', 0)); world = int(os.environ.get('WORLD_SIZE', 1))
    if world != 2 or torch.cuda.device_count() < 2:
        raise RuntimeError('Run distributed stages with torchrun and two GPUs')
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.cuda.set_device(int(os.environ['LOCAL_RANK']))
    dist.init_process_group('nccl', timeout=timedelta(minutes=30))
    return rank, torch.device('cuda', int(os.environ['LOCAL_RANK']))


def is_final(fold):
    return fold == 'final'


def split_for(fold, train, dev, config):
    if is_final(fold):
        return train, dev
    number = int(fold)
    if not 0 <= number < FOLDS:
        raise ValueError(fold)
    mapping = config['fold_map']
    eligible = [d for d in train if mapping[d['id']] != number]
    inner_ids = set()
    for domain in ('CL', 'IT'):
        rows = [d for d in eligible if d['domain'] == domain]
        ordered = sorted(rows, key=lambda d: hashlib.sha256(
            f'{SEED}:{number}:{d["id"]}'.encode()).hexdigest())
        inner_ids.update(d['id'] for d in ordered[:2])
    fitting = [d for d in eligible if d['id'] not in inner_ids]
    checking = [d for d in eligible if d['id'] in inner_ids]
    if len(fitting) != 60 or len(checking) != 4:
        raise RuntimeError('Expected 60 fitting, four inner-validation documents')
    return fitting, checking


def outer_for(fold, train, config):
    return [d for d in train if config['fold_map'][d['id']] == int(fold)]


def folder(fold):
    return ROOT / ('final' if is_final(fold) else f'fold_{int(fold):02d}')


def collate(rows, tokens, pad, device):
    seqs = [tokens[did][i] for did, i, _, _ in rows]
    width = max(len(s) for s in seqs)
    ids = torch.full((len(rows), width), pad, dtype=torch.long)
    mask = torch.zeros((len(rows), width), dtype=torch.bool)
    for j, seq in enumerate(seqs):
        ids[j, :len(seq)] = torch.as_tensor(seq, dtype=torch.long)
        mask[j, :len(seq)] = True
    y = torch.tensor([r[2] for r in rows], dtype=torch.long)
    domain = torch.tensor([r[3] for r in rows], dtype=torch.long)
    if device.type == 'cuda':
        return (ids.pin_memory().to(device, non_blocking=True),
                mask.pin_memory().to(device, non_blocking=True),
                y.pin_memory().to(device, non_blocking=True),
                domain.pin_memory().to(device, non_blocking=True))
    return ids.to(device), mask.to(device), y.to(device), domain.to(device)


def sentence_rows(docs, vocab):
    to_id = {v: i for i, v in enumerate(vocab)}
    return [(d['id'], i, to_id[y] if d['labels'] is not None else -1,
             0 if d['domain'] == 'CL' else 1)
            for d in docs for i, y in enumerate(d['labels'] or [None] * len(d['text']))]


def metrics(prob, docs, vocab):
    old = legacy()
    pred = {d['id']: [vocab[int(i)] for i in prob[d['id']].argmax(-1)] for d in docs}
    truth = {d['id']: d['labels'] for d in docs}
    present = [y for y in vocab if any(y in d['labels'] for d in docs)]
    all13 = old.prf(pred, truth, vocab)
    supported_f1 = sum(all13['per_label'][y]['f1'] for y in present) / len(present)
    domains = {}
    for domain in ('CL', 'IT'):
        subset = [d for d in docs if d['domain'] == domain]
        ps = {d['id']: pred[d['id']] for d in subset}
        ys = {d['id']: truth[d['id']] for d in subset}
        # Legacy PRF expects every predicted label in vocabulary.
        domains[domain] = old.prf(ps, ys, vocab)['macro_f1']
    return {'macro_all13': all13['macro_f1'], 'macro_supported': supported_f1,
            'domain_all13': domains, 'per_label': all13['per_label']}


class TargetEncoder(nn.Module):
    def __init__(self, base, labels_count):
        super().__init__()
        self.base = base
        width = base.hidden_size
        self.attention = nn.Linear(width, 1)
        self.pool = nn.Sequential(nn.Linear(width * 3, 512), nn.GELU(), nn.LayerNorm(512),
                                  nn.Dropout(.25))
        self.classifier = nn.Linear(512, labels_count)
        self.domain_bias = nn.Embedding(2, labels_count)
        nn.init.zeros_(self.domain_bias.weight)

    def forward(self, ids, mask, domain):
        base = self.base
        x = base.embedding_dropout(base.token_embedding(ids) * math.sqrt(base.hidden_size))
        visible = base._prefix_hybrid_visibility(mask, mask.sum(1))
        cosine, sine = base.rotary(ids.size(1), x.dtype)
        for block in base.blocks:
            x = checkpoint(block, x, visible, cosine, sine, use_reentrant=False) if self.training else block(x, visible, cosine, sine)
        x = base.final_norm(x)
        weights = self.attention(x).squeeze(-1).masked_fill(~mask, -1e4).softmax(-1)
        attention = torch.sum(x * weights.unsqueeze(-1), dim=1)
        mean = torch.sum(x * mask.unsqueeze(-1), dim=1) / mask.sum(1, keepdim=True).clamp_min(1)
        maximum = x.masked_fill(~mask.unsqueeze(-1), -1e4).max(dim=1).values
        emb = self.pool(torch.cat((attention, mean, maximum), dim=-1))
        logits = self.classifier(emb) + self.domain_bias(domain)
        return emb, logits


def new_encoder(old, device, vocab):
    base, tokenizer = old.load_base('03')
    base.output_bias.requires_grad_(False)  # LM output bias is unused for classification.
    return TargetEncoder(base, len(vocab)).to(device), tokenizer


def class_weights(docs, vocab, device):
    counts = Counter(y for d in docs for y in d['labels'])
    typical = sum(counts.values()) / len(vocab)
    values = [min(3., max(.6, math.sqrt(typical / max(1, counts[y])))) for y in vocab]
    scale = sum(values) / len(values)
    return torch.tensor([v / scale for v in values], device=device)


def gather_dict(local):
    received = [None, None]
    dist.all_gather_object(received, local)
    combined = {}
    for part in received:
        for did, value in part.items():
            if did in combined:
                raise RuntimeError('Duplicate distributed document: ' + did)
            combined[did] = value
    return combined


def autocast():
    return torch.autocast('cuda', dtype=torch.float16)


def finite_or_fail(loss):
    if not torch.isfinite(loss).all():
        raise RuntimeError('Non-finite loss; inspect checkpoint and batch')


@torch.no_grad()
def distributed_sentence_eval(model, docs, tokens, pad, vocab, rank, device, features=False):
    model.eval()
    own = {}
    for doc in docs[rank::2]:
        chunks = []
        logits = []
        for start in range(0, len(doc['text']), BATCH * 2):
            rows = [(doc['id'], i, -1, 0 if doc['domain'] == 'CL' else 1)
                    for i in range(start, min(start + BATCH * 2, len(doc['text'])))]
            ids, mask, _, domain = collate(rows, tokens, pad, device)
            with autocast():
                emb, score = model(ids, mask, domain)
            chunks.append(emb.float().cpu().half())
            logits.append(score.float().cpu())
        own[doc['id']] = {'features': torch.cat(chunks) if features else None,
                          'prob': torch.cat(logits).softmax(-1)}
    return own


def sentence_optimizer(model, steps):
    bottom, top, head = [], [], []
    half = len(model.base.blocks) // 2
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if not name.startswith('base.'):
            head.append(param)
        elif name.startswith('base.blocks.'):
            number = int(name.split('.')[2])
            (bottom if number < half else top).append(param)
        else:
            bottom.append(param)
    opt = torch.optim.AdamW([
        {'params': bottom, 'lr': 4e-6, 'initial_lr': 4e-6},
        {'params': top, 'lr': 1.2e-5, 'initial_lr': 1.2e-5},
        {'params': head, 'lr': 1.5e-4, 'initial_lr': 1.5e-4}], weight_decay=.04)
    warm = max(20, steps // 15)
    def schedule(step):
        if step < warm:
            return .1 + .9 * step / warm
        return .1 + .9 * .5 * (1 + math.cos(math.pi * min(1., (step-warm) / max(1, steps-warm))))
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, schedule)


def token_budget_batches(order, rows, tokens):
    batches = []
    current = []; widest = 0
    for index in order:
        row = rows[index]
        length = len(tokens[row[0]][row[1]])
        next_width = max(widest, length)
        if current and ((len(current)+1)*next_width > TOKEN_BUDGET or len(current) >= MAX_BATCH):
            batches.append(current); current = []; widest = 0
        current.append(index); widest = max(widest, length)
    if current:
        batches.append(current)
    return batches


def sentence_batches(rows, tokens, epoch, fold, rank):
    generator = torch.Generator().manual_seed(SEED + epoch * 101 +
                                              (0 if is_final(fold) else int(fold))*1009)
    counts = Counter((r[2], r[3]) for r in rows)
    sampling = torch.tensor([min(3., math.sqrt(len(rows) / (len(counts)*counts[(r[2],r[3])])))
                             for r in rows], dtype=torch.float32)
    selected = torch.multinomial(sampling, len(rows), replacement=True,
                                 generator=generator).tolist()
    selected.sort(key=lambda i: len(tokens[rows[i][0]][rows[i][1]]) // 32)
    buckets = [selected[i:i+BATCH*20] for i in range(0, len(selected), BATCH*20)]
    random.Random(SEED + epoch).shuffle(buckets)
    global_batches = [batch for bucket in buckets
                      for batch in token_budget_batches(bucket, rows, tokens)]
    if len(global_batches) % 2:
        global_batches.append(global_batches[-1])
    return global_batches[rank::2]


def train_sentence(fold):
    rank, device = init_dist()
    old, train, dev, _, vocab = sets()
    config = read_json(ROOT / 'config.json')
    fitting, checking = split_for(fold, train, dev, config)
    tokens = load_tokens()
    model, tokenizer = new_encoder(old, device, vocab)
    ddp = DDP(model, device_ids=[device.index], broadcast_buffers=False)
    rows = sentence_rows(fitting, vocab)
    val_rows = sentence_rows(checking, vocab)
    steps_per_epoch = math.ceil(len(sentence_batches(rows, tokens, 1, fold, rank)) / ACCUM)
    fixed_epochs = SENT_EPOCHS
    if is_final(fold):
        fold_epochs = [read_json(folder(str(f)) / 'sentence_status.json')['best_epoch']
                       for f in range(FOLDS) if (folder(str(f)) / 'sentence_status.json').exists()]
        fixed_epochs = max(3, min(SENT_EPOCHS, round(sum(fold_epochs) / len(fold_epochs)))) if fold_epochs else 6
    opt, scheduler = sentence_optimizer(model, steps_per_epoch * fixed_epochs)
    scaler = torch.cuda.amp.GradScaler()
    weights = class_weights(fitting, vocab, device)
    path = folder(fold); path.mkdir(parents=True, exist_ok=True)
    latest = path / 'sentence_latest.pt'; best_path = path / 'sentence_best.pt'
    first_epoch, first_batch, step, best, best_epoch, bad = 1, 0, 0, -1., 0, 0
    if latest.exists():
        ck = torch.load(latest, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
        first_epoch, first_batch, step = ck['epoch'], ck['next_batch'], ck['step']
        best, best_epoch, bad = ck['best'], ck['best_epoch'], ck['bad']
        if ck.get('rng') is not None:
            torch.set_rng_state(ck['rng'])
    start_time = time.time()
    last_snapshot = start_time
    deadline = float(os.environ.get('CS781_HIER_DEADLINE', time.time() + 20*3600))

    def save_latest(epoch, next_batch):
        if rank == 0:
            atomic_torch(latest, {'model': model.state_dict(), 'optimizer': opt.state_dict(),
                                  'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                                  'rng': torch.get_rng_state(), 'epoch': epoch,
                                  'next_batch': next_batch, 'step': step, 'best': best,
                                  'best_epoch': best_epoch, 'bad': bad})

    for epoch in range(first_epoch, fixed_epochs + 1):
        model.train()
        batches = sentence_batches(rows, tokens, epoch, fold, rank)
        opt.zero_grad(set_to_none=True)
        running = 0.
        for bi in range(first_batch if epoch == first_epoch else 0, len(batches)):
            group = [rows[i] for i in batches[bi]]
            ids, mask, y, domain = collate(group, tokens, int(tokenizer.pad_token_id), device)
            update = (bi+1) % ACCUM == 0 or bi+1 == len(batches)
            context = nullcontext() if update else ddp.no_sync()
            with context:
                with autocast():
                    _, logits = ddp(ids, mask, domain)
                    loss = F.cross_entropy(logits.float(), y, weight=weights,
                                           label_smoothing=.025) / ACCUM
                finite_or_fail(loss)
                scaler.scale(loss).backward()
            running += float(loss.detach()) * ACCUM
            if update:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                scaler.step(opt); scaler.update(); scheduler.step(); opt.zero_grad(set_to_none=True)
                step += 1
                if step % 150 == 0:
                    save_latest(epoch, bi+1)
            if rank == 0 and bi % 80 == 0:
                print(f'sentence {fold} epoch {epoch} batch {bi}/{len(batches)} loss {running/(bi+1):.4f} '
                      f'GPU {torch.cuda.memory_allocated(device)/2**30:.2f}GiB', flush=True)
        dist.barrier()
        own = distributed_sentence_eval(model, checking, tokens, int(tokenizer.pad_token_id),
                                         vocab, rank, device)
        prob = gather_dict({did: value['prob'] for did, value in own.items()})
        if rank == 0:
            report = metrics(prob, checking, vocab)
            score = report['macro_supported']
            improved = is_final(fold) or score > best + .002
            if improved:
                best, best_epoch, bad = score, epoch, 0
                atomic_torch(best_path, {'model': model.state_dict(), 'epoch': epoch,
                                         'supported_f1': best, 'all13_f1': report['macro_all13']})
            else:
                bad += 1
            atomic_json(path / 'sentence_status.json', {'best_epoch': best_epoch,
                        'best_supported_f1': best, 'best_path': str(best_path),
                        'current_epoch': epoch, 'validation': report,
                        'complete': False, 'elapsed_seconds': time.time()-start_time})
            print(f'sentence {fold} epoch {epoch} val supported macro F1={score:.4f}, '
                  f'all13={report["macro_all13"]:.4f}, best={best:.4f}', flush=True)
        dist.barrier()
        state = read_json(path / 'sentence_status.json')
        best, best_epoch = state['best_supported_f1'], state['best_epoch']
        bad = epoch - best_epoch
        save_latest(epoch+1, 0)
        dist.barrier()
        if rank == 0 and time.time()-last_snapshot >= 35*60:
            snapshot(); last_snapshot = time.time()
        dist.barrier()
        cutoff = (not is_final(fold) and epoch >= 4 and bad >= 3) or epoch >= fixed_epochs
        cutoff |= (time.time()-start_time > 55*60 and epoch >= 3)
        cutoff |= time.time() > deadline-3600
        if cutoff:
            break
    if rank == 0:
        state = read_json(path / 'sentence_status.json')
        state['complete'] = True
        atomic_json(path / 'sentence_status.json', state)
        # A finished fold resumes from its best model; the much larger Adam
        # state is only needed while the stage is unfinished.
        latest.unlink(missing_ok=True)
    dist.barrier(); dist.destroy_process_group()


def extract_features(fold):
    rank, device = init_dist()
    old, train, dev, test, vocab = sets()
    tokens = load_tokens()
    path = folder(fold)
    model, tokenizer = new_encoder(old, device, vocab)
    checkpoint_data = torch.load(path / 'sentence_best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint_data['model'])
    # Each GPU owns whole documents; rank files avoid a large NCCL object gather.
    for split, docs in (('train', train), ('dev', dev), ('test', test)):
        target = path / f'features_{split}.pt'
        if target.exists():
            continue
        own = distributed_sentence_eval(model, docs, tokens, int(tokenizer.pad_token_id),
                                         vocab, rank, device, features=True)
        shard = path / f'features_{split}_rank{rank}.pt'
        atomic_torch(shard, own)
        dist.barrier()
        if rank == 0:
            combined = {}
            for r in (0, 1):
                part = torch.load(path / f'features_{split}_rank{r}.pt', map_location='cpu', weights_only=False)
                combined.update(part)
            if set(combined) != {d['id'] for d in docs}:
                raise RuntimeError('Feature extraction missed documents')
            atomic_torch(target, combined)
            for r in (0, 1):
                (path / f'features_{split}_rank{r}.pt').unlink()
        dist.barrier()
    dist.destroy_process_group()


class DocumentModel(nn.Module):
    """Overlapping local attention plus a document-level sequence of window summaries."""
    def __init__(self, labels_count):
        super().__init__()
        self.position = nn.Embedding(32, 512)
        self.domain = nn.Embedding(2, 512)
        layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, dim_feedforward=1536,
                                           dropout=.18, activation='gelu', batch_first=True,
                                           norm_first=True)
        self.local = nn.TransformerEncoder(layer, num_layers=3, enable_nested_tensor=False)
        self.global_context = nn.GRU(512, 256, num_layers=2, bidirectional=True,
                                     batch_first=True, dropout=.15)
        self.normal = nn.LayerNorm(512)
        self.head_cl = nn.Sequential(nn.Dropout(.25), nn.Linear(512, 512), nn.GELU(),
                                      nn.Dropout(.2), nn.Linear(512, labels_count))
        self.head_it = nn.Sequential(nn.Dropout(.25), nn.Linear(512, 512), nn.GELU(),
                                      nn.Dropout(.2), nn.Linear(512, labels_count))
        self.boundary = nn.Linear(1024, 1)

    def forward(self, embeddings, domain):
        n = embeddings.size(0)
        if n == 0:
            raise RuntimeError('Empty document')
        indices = torch.arange(n, device=embeddings.device)
        pos = (indices * 32 // n).clamp_max(31)
        x = embeddings + self.position(pos) + self.domain.weight[domain][None, :]
        starts = list(range(0, n, STRIDE))
        windows = []
        key_masks = []
        for start in starts:
            part = x[start:start+WINDOW]
            length = len(part)
            windows.append(F.pad(part, (0, 0, 0, WINDOW-length)))
            key_masks.append(torch.arange(WINDOW, device=x.device) >= length)
        bank = torch.stack(windows)
        key_mask = torch.stack(key_masks)
        local = self.local(bank, src_key_padding_mask=key_mask)
        lengths = (~key_mask).sum(-1).clamp_min(1)
        summaries = (local * (~key_mask).unsqueeze(-1)).sum(1) / lengths[:, None]
        global_states, _ = self.global_context(summaries[None])
        local = local + global_states[0, :, None, :]
        combined = torch.zeros_like(x)
        counts = x.new_zeros(n, 1)
        for row, start in enumerate(starts):
            ix = torch.arange(start, min(start+WINDOW, n), device=x.device)
            combined.index_add_(0, ix, local[row, :len(ix)])
            counts.index_add_(0, ix, torch.ones(len(ix), 1, device=x.device))
        combined = self.normal(combined / counts.clamp_min(1))
        logits = (self.head_cl if domain == 0 else self.head_it)(combined)
        boundary = self.boundary(torch.cat((combined[:-1], combined[1:]), -1)).squeeze(-1)
        return logits, boundary


def document_training_order(docs, epoch, rank, fold):
    rng = random.Random(SEED + epoch*53 + (0 if is_final(fold) else int(fold))*1000)
    cl = [d for d in docs if d['domain'] == 'CL']
    it = [d for d in docs if d['domain'] == 'IT']
    rng.shuffle(cl); rng.shuffle(it)
    mixed = []
    for i, pair in enumerate(zip(cl, it)):
        mixed.extend(pair if i % 2 == 0 else pair[::-1])
    if len(mixed) != len(docs):
        raise RuntimeError('Training domains are imbalanced unexpectedly')
    return mixed[rank::2]


@torch.no_grad()
def distributed_document_eval(model, docs, features, rank, device):
    model.eval(); own = {}
    for doc in docs[rank::2]:
        emb = features[doc['id']]['features'].float().to(device)
        domain = 0 if doc['domain'] == 'CL' else 1
        with autocast():
            logits, _ = model(emb, domain)
        own[doc['id']] = logits.float().softmax(-1).cpu()
    return gather_dict(own)


def train_document(fold):
    rank, device = init_dist()
    _, train, dev, _, vocab = sets()
    config = read_json(ROOT / 'config.json')
    fitting, checking = split_for(fold, train, dev, config)
    path = folder(fold)
    feature_train = torch.load(path / 'features_train.pt', map_location='cpu', weights_only=False)
    feature_check = (torch.load(path / 'features_dev.pt', map_location='cpu', weights_only=False)
                     if is_final(fold) else feature_train)
    model = DocumentModel(len(vocab)).to(device)
    ddp = DDP(model, device_ids=[device.index], find_unused_parameters=True,
              broadcast_buffers=False)
    weights = class_weights(fitting, vocab, device)
    to_id = {y: i for i, y in enumerate(vocab)}
    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-5, weight_decay=.05)
    scaler = torch.cuda.amp.GradScaler()
    fixed_epochs = HEAD_EPOCHS
    if is_final(fold):
        fold_epochs = [read_json(folder(str(f)) / 'head_status.json')['best_epoch']
                       for f in range(FOLDS) if (folder(str(f)) / 'head_status.json').exists()]
        fixed_epochs = max(5, min(HEAD_EPOCHS, round(sum(fold_epochs)/len(fold_epochs)))) if fold_epochs else 10
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=fixed_epochs)
    latest = path / 'head_latest.pt'; best_path = path / 'head_best.pt'
    first_epoch, best, best_epoch = 1, -1., 0
    if latest.exists():
        ck = torch.load(latest, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
        first_epoch, best, best_epoch = ck['epoch'], ck['best'], ck['best_epoch']
    start = time.time()
    last_snapshot = start
    deadline = float(os.environ.get('CS781_HIER_DEADLINE', time.time()+20*3600))
    for epoch in range(first_epoch, fixed_epochs+1):
        model.train(); optimizer.zero_grad(set_to_none=True)
        order = document_training_order(fitting, epoch, rank, fold)
        total = 0.
        for j, doc in enumerate(order):
            emb = feature_train[doc['id']]['features'].float().to(device)
            gold = torch.tensor([to_id[y] for y in doc['labels']], dtype=torch.long, device=device)
            domain = 0 if doc['domain'] == 'CL' else 1
            update = (j+1) % ACCUM == 0 or j+1 == len(order)
            context = nullcontext() if update else ddp.no_sync()
            with context:
                with autocast():
                    logits, boundary = ddp(emb, domain)
                    ce = F.cross_entropy(logits.float(), gold, weight=weights,
                                         label_smoothing=.015)
                    if len(gold) > 1:
                        changes = (gold[1:] != gold[:-1]).float()
                        positive = torch.tensor(min(8., max(1., (len(changes)-changes.sum().item()) /
                                                          max(1., changes.sum().item()))), device=device)
                        aux = F.binary_cross_entropy_with_logits(boundary.float(), changes,
                                                                 pos_weight=positive)
                    else:
                        aux = ce.new_zeros(())
                    loss = (ce + .12*aux) / ACCUM
                finite_or_fail(loss)
                scaler.scale(loss).backward()
            total += float(loss.detach())*ACCUM
            if update:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        scheduler.step(); dist.barrier()
        prob = distributed_document_eval(model, checking, feature_check, rank, device)
        if rank == 0:
            report = metrics(prob, checking, vocab)
            score = report['macro_supported']
            improved = (is_final(fold) or score > best+.002)
            if improved:
                best, best_epoch = score, epoch
                atomic_torch(best_path, {'model': model.state_dict(), 'epoch': epoch,
                                         'supported_f1': score, 'all13_f1': report['macro_all13']})
            atomic_json(path / 'head_status.json', {'best_epoch': best_epoch,
                        'best_supported_f1': best, 'current_epoch': epoch,
                        'validation': report, 'complete': False,
                        'elapsed_seconds': time.time()-start})
            atomic_torch(latest, {'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                                  'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                                  'epoch': epoch+1, 'best': best, 'best_epoch': best_epoch})
            print(f'document {fold} epoch {epoch}: train loss {total/len(order):.4f}, '
                  f'val supported macro F1 {score:.4f}, all13 {report["macro_all13"]:.4f}, '
                  f'GPU {torch.cuda.memory_allocated(device)/2**30:.2f}GiB', flush=True)
        dist.barrier()
        if rank == 0 and time.time()-last_snapshot >= 35*60:
            snapshot(); last_snapshot = time.time()
        dist.barrier()
        best_epoch = read_json(path / 'head_status.json')['best_epoch']
        stop = not is_final(fold) and epoch >= 5 and epoch-best_epoch >= 4
        stop |= epoch >= fixed_epochs or (time.time()-start > 25*60 and epoch >= 4)
        stop |= time.time() > deadline-3600
        if stop:
            break
    if rank == 0:
        state = read_json(path / 'head_status.json')
        state['complete'] = True
        atomic_json(path / 'head_status.json', state)
        latest.unlink(missing_ok=True)
    dist.barrier(); dist.destroy_process_group()


def predict_head(fold, split):
    rank, device = init_dist()
    _, train, dev, test, vocab = sets()
    docs = {'train': train, 'dev': dev, 'test': test}[split]
    path = folder(fold)
    features = torch.load(path / f'features_{split}.pt', map_location='cpu', weights_only=False)
    model = DocumentModel(len(vocab)).to(device)
    ck = torch.load(path / 'head_best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'])
    prob = distributed_document_eval(model, docs, features, rank, device)
    if rank == 0:
        atomic_torch(path / f'head_prob_{split}.pt', prob)
        if split in ('dev', 'train'):
            labelled = docs if split == 'dev' else (
                docs if is_final(fold) else outer_for(fold, train, read_json(ROOT / 'config.json')))
            report = metrics(prob, labelled, vocab)
            atomic_json(path / f'head_{split}_metrics.json', report)
    dist.barrier(); dist.destroy_process_group()


def prior_prob(split):
    target = ROOT / f'prior6_prob_{split}.pt'
    if target.exists():
        return torch.load(target, map_location='cpu', weights_only=False)
    old, _, dev, test, vocab = sets()
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model, tokenizer = old.load_selected(6, device)
    docs = dev if split == 'dev' else test
    prob = old.probabilities(model, docs, tokenizer, vocab, device)
    atomic_torch(target, prob)
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return prob


def finished_models():
    names = [str(f) for f in range(FOLDS) if (folder(str(f)) / 'head_best.pt').is_file()]
    if (folder('final') / 'head_best.pt').is_file():
        names.append('final')
    return names


def combine_new(split, names, docs):
    chosen = [f for f in names if (folder(f) / f'head_prob_{split}.pt').is_file()]
    if set(chosen) != set(names):
        raise FileNotFoundError(f'Missing {split} probabilities: {set(names)-set(chosen)}')
    parts = {f: torch.load(folder(f) / f'head_prob_{split}.pt', map_location='cpu', weights_only=False)
             for f in chosen}
    fold_names = [f for f in names if f != 'final']
    if not fold_names:
        return {d['id']: parts['final'][d['id']] for d in docs}, {'final': 1.0}
    weights = {f: (0.7 / len(fold_names) if 'final' in names else 1. / len(fold_names))
               for f in fold_names}
    if 'final' in names:
        weights['final'] = .3
    return {d['id']: sum(weights[f] * parts[f][d['id']] for f in names) for d in docs}, weights


def select():
    _, train, dev, _, vocab = sets()
    prior = prior_prob('dev')
    prior_report = metrics(prior, dev, vocab)
    names = [f for f in finished_models() if (folder(f) / 'head_prob_dev.pt').is_file()]
    folds = [f for f in names if f != 'final' and (folder(f) / 'head_prob_train.pt').is_file()]
    oof = {}
    oof_sentence = {}
    config = read_json(ROOT / 'config.json')
    for f in folds:
        part = torch.load(folder(f) / 'head_prob_train.pt', map_location='cpu', weights_only=False)
        encoded = torch.load(folder(f) / 'features_train.pt', map_location='cpu', weights_only=False)
        for doc in train:
            if config['fold_map'][doc['id']] == int(f):
                oof[doc['id']] = part[doc['id']]
                oof_sentence[doc['id']] = encoded[doc['id']]['prob']
    oof_docs = [d for d in train if d['id'] in oof]
    oof_report = metrics(oof, oof_docs, vocab) if oof_docs else None
    sentence_oof_report = metrics(oof_sentence, oof_docs, vocab) if oof_docs else None
    selected = 'prior6'; all_reports = {'prior6': prior_report}
    weights = {}
    if names:
        new, weights = combine_new('dev', names, dev)
        blend = {d['id']: .75*new[d['id']] + .25*prior[d['id']] for d in dev}
        all_reports['new'] = metrics(new, dev, vocab)
        all_reports['blend'] = metrics(blend, dev, vocab)
        # A fixed, predeclared gate limits overfitting to the ten-document dev
        # set. No weight/transition grid is searched on development labels.
        oof_gain = (oof_report is not None and len(folds) >= 3 and
                    oof_report['macro_supported'] >= sentence_oof_report['macro_supported'] + .01)
        domain_ok = all(all_reports['new']['domain_all13'][domain] >=
                        prior_report['domain_all13'][domain] - .035 for domain in ('CL', 'IT'))
        candidates = []
        if oof_gain and domain_ok:
            if all_reports['new']['macro_supported'] >= prior_report['macro_supported'] + .025:
                candidates.append('new')
            if all_reports['blend']['macro_supported'] >= prior_report['macro_supported'] + .015:
                candidates.append('blend')
        if candidates:
            selected = max(candidates, key=lambda c: all_reports[c]['macro_supported'])
    record = {'selected': selected, 'models': names, 'folds_with_oof': folds,
              'new_model_weights': weights, 'development': all_reports,
              'oof': oof_report, 'sentence_oof': sentence_oof_report,
              'private_score_reference_prior6': .5058,
              'selection_rule': 'Fixed OOF gain, both-domain audit, and dev gain thresholds; no dev grid search.'}
    atomic_json(ROOT / 'selection.json', record)
    print('Selection:', selected, 'prior dev supported', prior_report['macro_supported'],
          'new dev supported', all_reports.get('new', {}).get('macro_supported'),
          'OOF supported', None if oof_report is None else oof_report['macro_supported'], flush=True)


def finalize():
    _, _, _, test, vocab = sets()
    selection = read_json(ROOT / 'selection.json')
    chosen = selection['selected']
    prior = prior_prob('test') if chosen in ('prior6', 'blend') else None
    if chosen in ('new', 'blend'):
        new, _ = combine_new('test', selection['models'], test)
        prob = (new if chosen == 'new' else
                {d['id']: .75*new[d['id']] + .25*prior[d['id']] for d in test})
    else:
        prob = prior
    prediction = {d['id']: [vocab[int(i)] for i in prob[d['id']].argmax(-1)] for d in test}
    if set(prediction) != {d['id'] for d in test} or any(
            len(prediction[d['id']]) != len(d['text']) for d in test):
        raise RuntimeError('Submission IDs or sentence counts do not match official test data')
    if any(y not in vocab for values in prediction.values() for y in values):
        raise RuntimeError('Unknown submission label')
    atomic_json(ROOT / 'prediction.json', prediction)
    temp = SUBMISSION.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ROOT / 'prediction.json', 'prediction.json')
    os.replace(temp, SUBMISSION)
    with zipfile.ZipFile(SUBMISSION) as z:
        if z.namelist() != ['prediction.json'] or json.loads(z.read('prediction.json')) != prediction:
            raise RuntimeError('Invalid submission ZIP')
    shutil.copyfile(SUBMISSION, WORK / 'submission.zip')
    shutil.copyfile(SUBMISSION, ROOT / 'submission.zip')
    atomic_json(ROOT / 'report.json', {'status': 'READY_FOR_MANUAL_UPLOAD',
                'submission': str(SUBMISSION), 'submission_sha256': digest(SUBMISSION),
                'selected': chosen, 'test_documents': len(test),
                'test_sentences': sum(len(d['text']) for d in test),
                'development_supported_macro_f1':
                selection['development'][chosen]['macro_supported'],
                'development_all13_macro_f1': selection['development'][chosen]['macro_all13'],
                'private_score': None,
                'note': '0.8 is a target, not a guaranteed or measured private score.'})
    snapshot()
    print('Submission ready:', SUBMISSION, flush=True)
    print('Recovery ready:', ARTIFACT, flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('action', choices=('prepare', 'sentence', 'extract', 'head',
                                           'predict-head', 'select', 'finalize', 'snapshot'))
    parser.add_argument('--fold', default='0')
    parser.add_argument('--split', choices=('train', 'dev', 'test'), default='dev')
    args = parser.parse_args()
    if args.action == 'prepare': prepare()
    elif args.action == 'sentence': train_sentence(args.fold)
    elif args.action == 'extract': extract_features(args.fold)
    elif args.action == 'head': train_document(args.fold)
    elif args.action == 'predict-head': predict_head(args.fold, args.split)
    elif args.action == 'select': select()
    elif args.action == 'finalize': finalize()
    else: snapshot()


if __name__ == '__main__':
    main()



In [ ]:
%%writefile /kaggle/working/cs781_phase3_context_joint_runner.py
"""Phase 3 context-aware joint encoder/document experiment.

This file and hierarchical_runner.py are embedded in the Run All notebook.
Only the supplied Phase 1 checkpoint and official Phase 3 data are used.
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
from torch import nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.checkpoint import checkpoint

import cs781_phase3_hierarchical_runner as u

WORK = Path('/kaggle/working')
ROOT = WORK / 'CS781_Capstone/phase3_joint'
ARTIFACT = WORK / 'CS781_Phase3_Context_Joint_All_Artifacts.zip'
SUBMISSION = WORK / 'CS781_Phase3_Context_Joint_Submission.zip'
u.ROOT = ROOT
u.ARTIFACT = ARTIFACT
u.OLD_ARCHIVES = ('CS781_Phase3_All_Artifacts.zip',
                  'CS781_Phase3_Hierarchical_All_Artifacts.zip',
                  'CS781_Phase3_One_All_Artifacts.zip')

SEED = 781230
MAX_TOKENS = 448
TARGET_LIMIT = 256
BASE_WINDOW, BASE_STRIDE = 8, 6
JOINT_WINDOW, JOINT_STRIDE = 16, 12
ACCUM = 2
MAX_HOURS = 9.45
ARCHIVE_INTERVAL = 38 * 60
SAVE_STEPS = 100
ROLE_GROUPS = {
    'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2,
    'ArgumentRespondent': 2, 'Statute': 3,
    'PrecedentReliedUpon': 3, 'PrecedentNotReliedUpon': 3,
    'PrecedentOverruled': 3, 'RatioOfTheDecision': 4,
    'RulingByLowerCourt': 4, 'RulingByPresentCourt': 4,
    'Dissent': 4, 'None': 5,
}


def log(*args):
    print(time.strftime('%Y-%m-%d %H:%M:%S'), *args, flush=True)


def setting():
    return {'version': 1, 'seed': SEED, 'max_tokens': MAX_TOKENS,
            'target_limit': TARGET_LIMIT, 'base_window': BASE_WINDOW,
            'joint_window': JOINT_WINDOW, 'accum': ACCUM,
            'phase1_base': '03', 'prior_reference': 'slot_06',
            'runner_sha256': u.digest(__file__)}


def stable_seed(value):
    random.seed(value)
    np.random.seed(value % (2**32 - 1))
    torch.manual_seed(value)
    torch.cuda.manual_seed_all(value)


def prepare():
    ROOT.mkdir(parents=True, exist_ok=True)
    u.restore()
    old, train, dev, test, vocab = u.sets()
    if set(vocab) != set(ROLE_GROUPS) or len(vocab) != 13:
        raise RuntimeError('Unexpected Phase 3 label vocabulary')
    config_path = ROOT / 'config.json'
    config = setting()
    config['fold_map'] = u.fold_map(train)
    if config_path.exists():
        existing = u.read_json(config_path)
        if existing != config:
            raise RuntimeError('Recovery configuration differs from this notebook')
    else:
        u.atomic_json(config_path, config)
    shutil.copy2(Path(u.__file__), ROOT / 'recovery_utilities.py')
    shutil.copy2(Path(__file__), ROOT / 'context_joint_runner.py')
    token_path = ROOT / 'tokens.pt'
    baseline_path = ROOT / 'baseline_tokens.pt'
    tokenizer = None
    if not token_path.exists() or not baseline_path.exists():
        _, tokenizer = old.load_base('03')
    if not token_path.exists():
        tokens = make_tokens(train + dev + test, tokenizer)
        u.atomic_torch(token_path, tokens)
        lengths = [len(row[0]) for rows in tokens.values() for row in rows]
        repeated = {}
        for d in train:
            for text, label in zip(d['text'], d['labels']):
                repeated.setdefault(' '.join(text.lower().split()), set()).add(label)
        labelled_docs = {label: sum(label in d['labels'] for d in train) for label in vocab}
        u.atomic_json(ROOT / 'input_audit.json', {
            'train_docs': len(train), 'dev_docs': len(dev), 'test_docs': len(test),
            'train_sentences': sum(len(d['text']) for d in train),
            'dev_sentences': sum(len(d['text']) for d in dev),
            'test_sentences': sum(len(d['text']) for d in test),
            'token_length_percentiles': np.percentile(lengths, [50, 90, 99, 100]).tolist(),
            'training_label_counts': dict(Counter(y for d in train for y in d['labels'])),
            'training_documents_per_label': labelled_docs,
            'sentence_strings_with_conflicting_training_roles':
                sum(len(roles) > 1 for roles in repeated.values()),
            'vocabulary': vocab,
        })
    if not baseline_path.exists():
        baseline = {}
        for doc in train + dev + test:
            baseline[doc['id']] = []
            for i in range(len(doc['text'])):
                ids = old.encode_sentence(tokenizer, doc, i)
                baseline[doc['id']].append((ids, max(0, len(ids) - 1), len(ids)))
        u.atomic_torch(baseline_path, baseline)
    log('Prepared official documents and target-aware token cache:', token_path)
    if not ARTIFACT.exists():
        u.snapshot()


def make_tokens(docs, tokenizer):
    encode = lambda s: tokenizer.backend.encode(s, add_special_tokens=False).ids
    marker = encode('\n[TARGET SENTENCE] ')
    separator = encode(' [SENTENCE] ')
    if not marker:
        raise RuntimeError('Tokenizer produced no target marker tokens')
    out = {}
    for doc in docs:
        texts = doc['text']
        full = [encode(t) for t in texts]
        rows = []
        for i, target in enumerate(full):
            if len(target) > TARGET_LIMIT:
                front = TARGET_LIMIT * 3 // 4
                target = target[:front] + target[-(TARGET_LIMIT - front):]
            if not target:
                target = [int(tokenizer.pad_token_id)]
            earlier = full[max(0, i-2):i]
            later = full[i+1:i+3]
            left = encode(' [EARLIER CONTEXT] ')
            for segment in earlier:
                left += separator + segment
            right = encode(' [LATER CONTEXT] ')
            for segment in later:
                right += separator + segment
            budget = MAX_TOKENS - len(marker) - len(target)
            left_budget = min(len(left), budget // 2)
            right_budget = min(len(right), budget - left_budget)
            left_budget = min(len(left), budget - right_budget)
            left_part = left[-left_budget:] if left_budget else []
            right_part = right[:right_budget]
            start = len(left_part) + len(marker)
            ids = left_part + marker + target + right_part
            if not 0 < start < start + len(target) <= len(ids) <= MAX_TOKENS:
                raise RuntimeError('Bad target span encoding')
            rows.append((ids, start, start + len(target)))
        out[doc['id']] = rows
    return out


def windows(doc, size, stride):
    n = len(doc['text'])
    if n == 0:
        raise RuntimeError('Empty official document')
    start = 0
    while True:
        end = min(n, start + size)
        yield (doc['id'], start, end)
        if end == n:
            break
        start += stride


def batch_window(item, docs_by_id, tokens, pad, label_to_id, device):
    did, start, end = item
    doc = docs_by_id[did]
    rows = tokens[did][start:end]
    width = max(len(r[0]) for r in rows)
    n = len(rows)
    ids = torch.full((n, width), pad, dtype=torch.long)
    mask = torch.zeros((n, width), dtype=torch.bool)
    target = torch.zeros((n, width), dtype=torch.bool)
    for j, (seq, a, b) in enumerate(rows):
        ids[j, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        mask[j, :len(seq)] = True
        target[j, a:b] = True
    y = torch.tensor([label_to_id[x] for x in doc['labels'][start:end]], dtype=torch.long) if doc['labels'] else None
    pos = torch.tensor([min(31, int(32 * i / max(1, len(doc['text']))))
                        for i in range(start, end)], dtype=torch.long)
    def gpu(t):
        return t.pin_memory().to(device, non_blocking=True) if device.type == 'cuda' else t.to(device)
    return (gpu(ids), gpu(mask), gpu(target), gpu(pos),
            0 if doc['domain'] == 'CL' else 1, gpu(y) if y is not None else None)


class ContextJoint(nn.Module):
    def __init__(self, base, n_labels, mode):
        super().__init__()
        if mode not in {'baseline', 'joint'}:
            raise ValueError(mode)
        self.base, self.mode = base, mode
        self.base.output_bias.requires_grad_(False)
        width = base.hidden_size
        self.attention = nn.Linear(width, 1) if mode == 'joint' else None
        self.project = nn.Sequential(nn.Linear(width * (3 if mode == 'joint' else 2), 512),
                                     nn.LayerNorm(512), nn.GELU(), nn.Dropout(.16))
        self.position = nn.Embedding(32, 512)
        self.domain = nn.Embedding(2, 512)
        self.gru = nn.GRU(512, 256, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=.15)
        self.gru_norm = nn.LayerNorm(512)
        self.document_attention = nn.TransformerEncoderLayer(
            512, 8, dim_feedforward=1024, dropout=.12,
            activation='gelu', batch_first=True, norm_first=True) if mode == 'joint' else None
        self.document_norm = nn.LayerNorm(512) if mode == 'joint' else None
        self.cl_head = nn.Sequential(nn.Dropout(.2), nn.Linear(512, n_labels))
        self.it_head = nn.Sequential(nn.Dropout(.2), nn.Linear(512, n_labels))
        if mode == 'joint':
            self.coarse = nn.Linear(512, 6)
            self.argument = nn.Linear(512, 2)
            self.boundary = nn.Linear(1024, 1)

    def forward(self, ids, mask, target, pos, domain):
        base = self.base
        x = base.embedding_dropout(base.token_embedding(ids) * math.sqrt(base.hidden_size))
        visible = base._prefix_hybrid_visibility(mask, mask.sum(1))
        cosine, sine = base.rotary(ids.size(1), x.dtype)
        for block in base.blocks:
            x = checkpoint(block, x, visible, cosine, sine, use_reentrant=False) if self.training else block(x, visible, cosine, sine)
        x = base.final_norm(x)
        if self.mode == 'joint':
            att = self.attention(x).squeeze(-1).masked_fill(~target, -1e4).softmax(-1)
            pooled = (x * att.unsqueeze(-1)).sum(1)
            mean = (x * target.unsqueeze(-1)).sum(1) / target.sum(1, keepdim=True).clamp_min(1)
            maximum = x.masked_fill(~target.unsqueeze(-1), -1e4).max(1).values
            vectors = torch.cat((pooled, mean, maximum), dim=-1)
        else:
            last = x[torch.arange(x.size(0), device=x.device), mask.sum(1).long() - 1]
            mean = (x * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
            vectors = torch.cat((last, mean), dim=-1)
        h = self.project(vectors)
        h = h + self.position(pos) + self.domain(torch.tensor(domain, device=ids.device))
        recurrent, _ = self.gru(h[None])
        h = self.gru_norm(h + recurrent[0])
        if self.mode == 'joint':
            h = self.document_norm(h + self.document_attention(h[None])[0])
        fine = self.cl_head(h) if domain == 0 else self.it_head(h)
        if self.mode == 'baseline':
            return fine, None, None, None
        coarse = self.coarse(h)
        argument = self.argument(h)
        boundaries = self.boundary(torch.cat((h[:-1], h[1:]), -1)).squeeze(-1)
        return fine, coarse, argument, boundaries


def loss_weights(train, vocab, device):
    counts = Counter(y for d in train for y in d['labels'])
    typical = sum(counts.values()) / len(vocab)
    weights = [min(2.5, max(.65, math.sqrt(typical / max(1, counts[y])))) for y in vocab]
    scale = sum(weights) / len(weights)
    return torch.tensor([v / scale for v in weights], dtype=torch.float32, device=device)


def objective(output, gold, vocab, weights, mode):
    fine, coarse, arg, boundary = output
    loss = F.cross_entropy(fine.float(), gold, weight=weights)
    if mode == 'baseline':
        return loss
    groups = torch.tensor([ROLE_GROUPS[x] for x in vocab], device=gold.device)
    loss = loss + .16 * F.cross_entropy(coarse.float(), groups[gold])
    ids = {x: vocab.index(x) for x in ('ArgumentPetitioner', 'ArgumentRespondent')}
    is_arg = (gold == ids['ArgumentPetitioner']) | (gold == ids['ArgumentRespondent'])
    if is_arg.any():
        arg_gold = (gold[is_arg] == ids['ArgumentRespondent']).long()
        loss = loss + .12 * F.cross_entropy(arg[is_arg].float(), arg_gold)
    if gold.numel() > 1:
        changed = (gold[:-1] != gold[1:]).float()
        loss = loss + .08 * F.binary_cross_entropy_with_logits(boundary.float(), changed)
    return loss


def optimizer_for(model, total_updates):
    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith('base.blocks.'):
            layer = int(name.split('.')[2])
            lr = 5e-6 + (2.0e-5 - 5e-6) * layer / max(1, len(model.base.blocks) - 1)
        elif name.startswith('base.'):
            lr = 5e-6
        else:
            lr = 1.2e-4
        decay = 0. if name.endswith('bias') or 'norm' in name.lower() else .02
        key = (lr, decay)
        groups.setdefault(key, []).append(param)
    params = [{'params': ps, 'lr': lr, 'weight_decay': wd} for (lr, wd), ps in groups.items()]
    try:
        opt = torch.optim.AdamW(params, fused=True)
    except (TypeError, RuntimeError):
        opt = torch.optim.AdamW(params)
    warmup = max(10, int(.06 * total_updates))
    def schedule(step):
        if step < warmup:
            return max(.05, (step + 1) / warmup)
        fraction = min(1., (step - warmup) / max(1, total_updates - warmup))
        return .08 + .92 * .5 * (1 + math.cos(math.pi * fraction))
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, schedule)


def sample_gpus(stage, epoch, step):
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=index,utilization.gpu,memory.used,memory.total,power.draw',
             '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=15, check=True)
        rows = []
        for line in result.stdout.strip().splitlines():
            values = [x.strip() for x in line.split(',')]
            rows.append({'index': int(values[0]), 'utilization_percent': int(values[1]),
                         'memory_mib': int(values[2]), 'total_mib': int(values[3]),
                         'power_watts': float(values[4])})
    except Exception as exc:
        rows = [{'sampling_error': str(exc)}]
    sample = {'timestamp': time.time(), 'stage': stage, 'epoch': epoch,
              'optimizer_step': step, 'gpus': rows}
    with (ROOT / 'gpu_samples.jsonl').open('a', encoding='utf8') as f:
        f.write(json.dumps(sample) + '\n')


@torch.no_grad()
def evaluate(ddp, docs, docs_by_id, tokens, pad, vocab, mode, rank, device):
    ddp.eval()
    label_to_id = {x: i for i, x in enumerate(vocab)}
    size, stride = (BASE_WINDOW, BASE_STRIDE) if mode == 'baseline' else (JOINT_WINDOW, JOINT_STRIDE)
    local = {}
    for doc in docs[rank::2]:
        n = len(doc['text'])
        summed = torch.zeros((n, len(vocab)), dtype=torch.float32)
        counts = torch.zeros(n, dtype=torch.float32)
        for item in windows(doc, size, stride):
            ids, mask, target, pos, domain, _ = batch_window(
                item, docs_by_id, tokens, pad, label_to_id, device)
            with u.autocast():
                logits = ddp.module(ids, mask, target, pos, domain)[0]
            probability = logits.float().softmax(-1).cpu()
            summed[item[1]:item[2]] += probability
            counts[item[1]:item[2]] += 1
        if not torch.all(counts > 0):
            raise RuntimeError('Uncovered sentence in evaluation')
        local[doc['id']] = summed / counts[:, None]
    received = [None, None]
    dist.all_gather_object(received, local)
    merged = {did: p for part in received for did, p in part.items()}
    if set(merged) != {d['id'] for d in docs}:
        raise RuntimeError('Distributed evaluation missing a document')
    return merged


def report_predictions(prob, docs, vocab):
    report = u.metrics(prob, docs, vocab)
    pred = {did: [vocab[int(i)] for i in p.argmax(-1)] for did, p in prob.items()}
    report['predicted_label_counts'] = dict(Counter(y for labels in pred.values() for y in labels))
    report['gold_label_counts'] = dict(Counter(y for d in docs for y in d['labels']))
    confusions = Counter((gold, guess) for d in docs for gold, guess in zip(d['labels'], pred[d['id']]))
    report['confusion'] = {gold: {guess: confusions[(gold, guess)] for guess in vocab}
                           for gold in vocab}
    selected = ['ArgumentRespondent', 'PrecedentNotReliedUpon',
                'RulingByLowerCourt', 'RatioOfTheDecision']
    report['focus'] = {name: report['per_label'][name] for name in selected}
    return report


def stage_path(stage):
    return ROOT / stage


def stage_spec(stage, train, config):
    mode = 'baseline' if stage == 'pilot_baseline' else 'joint'
    fold = 0 if stage in {'pilot_baseline', 'pilot_joint'} else (
        1 if stage == 'confirm_1' else 2 if stage == 'confirm_2' else None)
    fitting = train if fold is None else [d for d in train if config['fold_map'][d['id']] != fold]
    heldout = [] if fold is None else [d for d in train if config['fold_map'][d['id']] == fold]
    if fold is None and stage != 'final':
        raise ValueError(stage)
    if fold is not None and (len(fitting) != 64 or len(heldout) != 16):
        raise RuntimeError('Fold split is not 64/16 documents')
    if stage == 'pilot_baseline':
        return mode, fitting, heldout, 10, 5, 3, 1.4
    if stage == 'pilot_joint':
        return mode, fitting, heldout, 22, 8, 5, 2.5
    pilot = u.read_json(stage_path('pilot_joint') / 'status.json')
    fixed = max(8, min(20, int(pilot['best_epoch'])))
    if stage == 'final':
        fixed = min(24, fixed + 2)
        return mode, fitting, heldout, fixed, fixed, 999, 2.0
    return mode, fitting, heldout, fixed, fixed, 999, 1.55


def run_stage(stage, deadline):
    rank, device = u.init_dist()
    try:
        stable_seed(SEED + rank + sum(ord(c) for c in stage))
        old, train, dev, _, vocab = u.sets()
        config = u.read_json(ROOT / 'config.json')
        mode, fitting, heldout, max_epochs, min_epochs, patience, cap_hours = stage_spec(stage, train, config)
        folder = stage_path(stage)
        folder.mkdir(parents=True, exist_ok=True)
        status_path = folder / 'status.json'
        if status_path.exists() and u.read_json(status_path).get('complete'):
            log(stage, 'already complete')
            return
        token_file = 'baseline_tokens.pt' if mode == 'baseline' else 'tokens.pt'
        tokens = torch.load(ROOT / token_file, map_location='cpu', weights_only=False)
        docs_by_id = {d['id']: d for d in train + dev}
        base, tokenizer = old.load_base('03')
        model = ContextJoint(base, len(vocab), mode).to(device)
        ddp = DDP(model, device_ids=[device.index], find_unused_parameters=True,
                  broadcast_buffers=False, gradient_as_bucket_view=True)
        pad = int(tokenizer.pad_token_id)
        size, stride = (BASE_WINDOW, BASE_STRIDE) if mode == 'baseline' else (JOINT_WINDOW, JOINT_STRIDE)
        items = [item for doc in fitting for item in windows(doc, size, stride)]
        updates_per_epoch = math.ceil(len(items) / (2 * ACCUM))
        optimizer, scheduler = optimizer_for(model, updates_per_epoch * max_epochs)
        scaler = torch.cuda.amp.GradScaler(enabled=True)
        weights = loss_weights(fitting, vocab, device)
        label_to_id = {x: i for i, x in enumerate(vocab)}
        latest = folder / 'latest.pt'
        best = folder / 'best.pt'
        state = {'epoch': 1, 'next_batch': 0, 'step': 0, 'best_f1': -1.,
                 'best_epoch': 0, 'bad_epochs': 0, 'complete': False}
        if latest.exists():
            ck = torch.load(latest, map_location='cpu', weights_only=False)
            model.load_state_dict(ck['model'], strict=True)
            optimizer.load_state_dict(ck['optimizer'])
            scheduler.load_state_dict(ck['scheduler'])
            scaler.load_state_dict(ck['scaler'])
            state.update(ck['state'])
            log(stage, 'resumed epoch', state['epoch'], 'batch', state['next_batch'])
        local_end = min(deadline - 22 * 60, time.time() + cap_hours * 3600)
        last_archive = time.time()

        def save_checkpoint():
            nonlocal last_archive
            dist.barrier()
            if rank == 0:
                u.atomic_torch(latest, {'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                    'scaler': scaler.state_dict(), 'state': state})
                if time.time() - last_archive > ARCHIVE_INTERVAL:
                    u.snapshot()
                    last_archive = time.time()
            dist.barrier()

        last_epoch = int(state['epoch']) - 1
        for epoch in range(int(state['epoch']), max_epochs + 1):
            rng = random.Random(SEED * 7 + epoch)
            order = list(items)
            rng.shuffle(order)
            while len(order) % (2 * ACCUM):
                order.append(order[len(order) % len(items)])
            assigned = order[rank::2]
            ddp.train()
            optimizer.zero_grad(set_to_none=True)
            start_batch = int(state['next_batch']) if epoch == state['epoch'] else 0
            epoch_start = time.time()
            loss_total, token_count = 0., 0
            for batch_i in range(start_batch, len(assigned)):
                ids, mask, target, pos, domain, gold = batch_window(
                    assigned[batch_i], docs_by_id, tokens, pad, label_to_id, device)
                sync = ((batch_i + 1) % ACCUM == 0)
                with (nullcontext() if sync else ddp.no_sync()):
                    with u.autocast():
                        outputs = ddp(ids, mask, target, pos, domain)
                        loss = objective(outputs, gold, vocab, weights, mode) / ACCUM
                    u.finite_or_fail(loss)
                    scaler.scale(loss).backward()
                loss_total += float(loss.detach()) * ACCUM
                token_count += int(mask.sum())
                if sync:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                    state['step'] += 1
                    state['next_batch'] = batch_i + 1
                    if state['step'] % 25 == 0 and rank == 0:
                        sample_gpus(stage, epoch, state['step'])
                    if state['step'] % SAVE_STEPS == 0:
                        save_checkpoint()
            dist.barrier()
            dev_prob = evaluate(ddp, dev, docs_by_id, tokens, pad, vocab, mode, rank, device)
            dev_report = report_predictions(dev_prob, dev, vocab)
            score = float(dev_report['macro_supported'])
            if rank == 0:
                u.atomic_json(folder / f'dev_epoch_{epoch:02d}.json', dev_report)
                if score > state['best_f1'] + 1e-5:
                    state['best_f1'] = score
                    state['best_epoch'] = epoch
                    state['bad_epochs'] = 0
                    u.atomic_torch(best, {'model': model.state_dict(), 'epoch': epoch,
                                          'dev_report': dev_report, 'mode': mode})
                else:
                    state['bad_epochs'] += 1
                entry = {'epoch': epoch, 'optimizer_steps': state['step'],
                         'train_loss_rank0': loss_total / max(1, len(assigned) - start_batch),
                         'rank0_tokens_per_second': token_count / max(1, time.time() - epoch_start),
                         'epoch_seconds': time.time() - epoch_start,
                         'dev_macro_supported': score,
                         'dev_macro_all13': dev_report['macro_all13'],
                         'dev_domain_all13': dev_report['domain_all13'],
                         'dev_focus_f1': {name: value['f1']
                                          for name, value in dev_report['focus'].items()},
                         'gpu_allocated_mib_rank0': torch.cuda.memory_allocated(device) / 2**20,
                         'gpu_reserved_mib_rank0': torch.cuda.memory_reserved(device) / 2**20}
                with (folder / 'history.jsonl').open('a', encoding='utf8') as f:
                    f.write(json.dumps(entry) + '\n')
                log(stage, json.dumps(entry))
                sample_gpus(stage, epoch, state['step'])
            state_list = [state if rank == 0 else None]
            dist.broadcast_object_list(state_list, src=0)
            state = state_list[0]
            state['epoch'] = epoch + 1
            state['next_batch'] = 0
            save_checkpoint()
            if rank == 0 and time.time() - last_archive > ARCHIVE_INTERVAL:
                u.snapshot()
                last_archive = time.time()
            dist.barrier()
            early = epoch >= min_epochs and state['bad_epochs'] >= patience
            required_before_time_stop = min_epochs if stage.startswith('pilot_') else 3
            cap = time.time() >= local_end and epoch >= required_before_time_stop
            if early or cap:
                if rank == 0:
                    log(stage, 'stopping:', 'patience' if early else 'time cap')
                last_epoch = epoch
                break
            last_epoch = epoch

        dist.barrier()
        chosen = best if stage.startswith('pilot_') else latest
        ck = torch.load(chosen, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model'], strict=True)
        if heldout:
            outer = evaluate(ddp, heldout, docs_by_id, tokens, pad, vocab, mode, rank, device)
            outer_report = report_predictions(outer, heldout, vocab)
        else:
            outer, outer_report = None, None
        dev_best = evaluate(ddp, dev, docs_by_id, tokens, pad, vocab, mode, rank, device)
        if rank == 0:
            if outer is not None:
                u.atomic_torch(folder / 'outer_prob.pt', outer)
                u.atomic_json(folder / 'outer_metrics.json', outer_report)
            u.atomic_torch(folder / 'dev_prob.pt', dev_best)
            u.atomic_json(folder / 'selected_dev_metrics.json',
                          report_predictions(dev_best, dev, vocab))
            u.atomic_torch(folder / 'selected.pt', {'model': model.state_dict(),
                           'epoch': last_epoch, 'mode': mode})
            state['complete'] = True
            state['selected_checkpoint'] = 'selected.pt'
            state['time_limited'] = bool(time.time() >= local_end and last_epoch < max_epochs)
            state['trained_epochs'] = last_epoch
            state['outer_supported'] = outer_report['macro_supported'] if outer_report else None
            u.atomic_json(status_path, state)
            latest.unlink(missing_ok=True)
            best.unlink(missing_ok=True)
            u.snapshot()
            log(stage, 'complete; outer supported F1', state['outer_supported'])
        dist.barrier()
    finally:
        if dist.is_initialized():
            dist.destroy_process_group()


def gate():
    _, train, dev, _, vocab = u.sets()
    baseline_status = u.read_json(stage_path('pilot_baseline') / 'status.json')
    joint_status = u.read_json(stage_path('pilot_joint') / 'status.json')
    base = u.read_json(stage_path('pilot_baseline') / 'outer_metrics.json')
    joint = u.read_json(stage_path('pilot_joint') / 'outer_metrics.json')
    gain = joint['macro_supported'] - base['macro_supported']
    per_domain = {d: joint['domain_all13'][d] - base['domain_all13'][d]
                  for d in ('CL', 'IT')}
    focus = {name: joint['per_label'][name]['f1'] - base['per_label'][name]['f1']
             for name in ('ArgumentRespondent', 'PrecedentNotReliedUpon',
                          'RulingByLowerCourt', 'RatioOfTheDecision')}
    enough_training = (baseline_status['trained_epochs'] >= 5 and
                       joint_status['trained_epochs'] >= 8)
    passed = enough_training and gain >= .03 and min(per_domain.values()) >= -.025 and (
        sum(1 for delta in focus.values() if delta > 0) >= 2)
    decision = {'pass': passed, 'required_supported_macro_gain': .03,
                'minimum_training_met': enough_training,
                'baseline_epochs': baseline_status['trained_epochs'],
                'joint_epochs': joint_status['trained_epochs'],
                'measured_gain': gain, 'per_domain_all13_delta': per_domain,
                'focus_label_f1_delta': focus, 'baseline_outer': base,
                'joint_outer': joint,
                'note': 'This pilot uses exactly the same 16 held-out training documents for both models.'}
    u.atomic_json(ROOT / 'pilot_gate.json', decision)
    log('Pilot gate', 'PASS' if passed else 'STOP', 'supported F1 gain', round(gain, 4))
    u.snapshot()
    return passed


def infer_stage(stage, split):
    rank, device = u.init_dist()
    try:
        stable_seed(SEED + rank)
        old, train, dev, test, vocab = u.sets()
        docs = dev if split == 'dev' else test
        token_file = 'baseline_tokens.pt' if mode == 'baseline' else 'tokens.pt'
        tokens = torch.load(ROOT / token_file, map_location='cpu', weights_only=False)
        base, tokenizer = old.load_base('03')
        mode = 'baseline' if stage == 'pilot_baseline' else 'joint'
        model = ContextJoint(base, len(vocab), mode).to(device)
        ck = torch.load(stage_path(stage) / 'selected.pt', map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model'], strict=True)
        ddp = DDP(model, device_ids=[device.index], find_unused_parameters=True,
                  broadcast_buffers=False)
        docs_by_id = {d['id']: d for d in train + dev + test}
        prob = evaluate(ddp, docs, docs_by_id, tokens, int(tokenizer.pad_token_id),
                        vocab, mode, rank, device)
        if rank == 0:
            u.atomic_torch(stage_path(stage) / f'{split}_prob.pt', prob)
            if split == 'dev':
                u.atomic_json(stage_path(stage) / 'selected_dev_metrics.json',
                              report_predictions(prob, docs, vocab))
            log(stage, split, 'inference complete')
        dist.barrier()
    finally:
        if dist.is_initialized():
            dist.destroy_process_group()


def combine(prob_a, prob_b, weight_a):
    if set(prob_a) != set(prob_b):
        raise RuntimeError('Probability document IDs differ')
    return {did: weight_a * prob_a[did] + (1 - weight_a) * prob_b[did]
            for did in prob_a}


def choose():
    _, train, dev, _, vocab = u.sets()
    prior = u.prior_prob('dev')
    prior_report = report_predictions(prior, dev, vocab)
    result = {'prior6_dev': prior_report, 'pilot_gate': u.read_json(ROOT / 'pilot_gate.json')}
    stages = ['pilot_joint', 'confirm_1', 'confirm_2']
    complete = all((stage_path(s) / 'status.json').is_file() and
                   u.read_json(stage_path(s) / 'status.json').get('complete') for s in stages)
    final_status_path = stage_path('final') / 'status.json'
    final_ready = (final_status_path.is_file() and
                   u.read_json(final_status_path).get('complete') and
                   u.read_json(final_status_path).get('trained_epochs', 0) >= 8)
    if not result['pilot_gate']['pass'] or not complete or not final_ready:
        result.update({'selected': 'prior6', 'reason': 'Pilot gate failed or full fit incomplete',
                       'private_score': None})
        u.atomic_json(ROOT / 'selection.json', result)
        return result

    # Pool the disjoint held-out fold predictions once. Folds 3-4 remain unused.
    mapping = u.read_json(ROOT / 'config.json')['fold_map']
    oof = {}
    for number, stage in enumerate(stages):
        part = torch.load(stage_path(stage) / 'outer_prob.pt', map_location='cpu', weights_only=False)
        expected = {d['id'] for d in train if mapping[d['id']] == number}
        if set(part) != expected or set(part) & set(oof):
            raise RuntimeError('Outer fold leakage or missing predictions')
        oof.update(part)
    covered = [d for d in train if d['id'] in oof]
    oof_report = report_predictions(oof, covered, vocab)
    result['joint_heldout_48docs'] = oof_report
    result['confirm_folds'] = {
        s: u.read_json(stage_path(s) / 'outer_metrics.json')['macro_supported']
        for s in stages}
    joint = torch.load(stage_path('final') / 'dev_prob.pt', map_location='cpu', weights_only=False)
    joint_report = report_predictions(joint, dev, vocab)
    blend = combine(joint, prior, .8)
    blend_report = report_predictions(blend, dev, vocab)
    result['joint_dev'] = joint_report
    result['fixed_80joint_20prior_dev'] = blend_report

    def domain_safe(candidate, margin):
        return all(candidate['domain_all13'][d] >= prior_report['domain_all13'][d] - margin
                   for d in ('CL', 'IT'))

    # Only two predeclared choices; the official dev set is too small for a grid search.
    if joint_report['macro_supported'] >= prior_report['macro_supported'] + .01 and domain_safe(joint_report, .03):
        selected, reason = 'joint', 'Joint final model beat the prior on dev and passed the domain guard'
    elif blend_report['macro_supported'] >= prior_report['macro_supported'] + .015 and domain_safe(blend_report, .025):
        selected, reason = 'fixed_blend', 'Predeclared blend beat the prior on dev and passed the domain guard'
    elif (oof_report['macro_supported'] >= .48 and
          joint_report['macro_supported'] >= prior_report['macro_supported'] - .005 and
          domain_safe(joint_report, .02)):
        selected, reason = 'joint', 'Held-out evidence passed and dev was approximately tied'
    else:
        selected, reason = 'prior6', 'New model did not clear the conservative dev guard'
    result.update({'selected': selected, 'reason': reason, 'private_score': None})
    u.atomic_json(ROOT / 'selection.json', result)
    u.snapshot()
    log('Selection', selected, reason)
    return result


def package():
    _, _, dev, test, vocab = u.sets()
    decision = u.read_json(ROOT / 'selection.json')
    selected = decision['selected']
    old_prob = u.prior_prob('test') if selected in {'prior6', 'fixed_blend'} else None
    if selected == 'prior6':
        prob = old_prob
    else:
        new_prob = torch.load(stage_path('final') / 'test_prob.pt', map_location='cpu', weights_only=False)
        prob = new_prob if selected == 'joint' else combine(new_prob, old_prob, .8)
    prediction = {d['id']: [vocab[int(i)] for i in prob[d['id']].argmax(-1)] for d in test}
    expected = {d['id']: len(d['text']) for d in test}
    if set(prediction) != set(expected) or any(len(prediction[did]) != n for did, n in expected.items()):
        raise RuntimeError('Submission document IDs or sentence counts are wrong')
    if sum(map(len, prediction.values())) != 2197:
        raise RuntimeError('Unexpected Phase 3 test sentence count')
    if any(y not in vocab for labels in prediction.values() for y in labels):
        raise RuntimeError('Submission has an unknown role label')
    u.atomic_json(ROOT / 'prediction.json', prediction)
    temp = SUBMISSION.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ROOT / 'prediction.json', 'prediction.json')
    os.replace(temp, SUBMISSION)
    with zipfile.ZipFile(SUBMISSION) as z:
        if z.namelist() != ['prediction.json'] or json.loads(z.read('prediction.json')) != prediction:
            raise RuntimeError('ZIP verification failed')
    shutil.copy2(SUBMISSION, WORK / 'submission.zip')
    shutil.copy2(SUBMISSION, ROOT / 'submission.zip')
    prior_prediction = {d['id']: [vocab[int(i)] for i in u.prior_prob('test')[d['id']].argmax(-1)]
                        for d in test}
    changes = sum(a != b for did in prediction for a, b in zip(prediction[did], prior_prediction[did]))
    report = {'status': 'READY_FOR_MANUAL_UPLOAD', 'selected': selected,
              'selection_reason': decision['reason'], 'submission': str(SUBMISSION),
              'submission_sha256': u.digest(SUBMISSION), 'test_documents': len(test),
              'test_sentences': sum(expected.values()), 'labels_changed_vs_slot06': changes,
              'prior_private_macro_f1': .5058, 'current_private_macro_f1': None,
              'target_private_macro_f1': .8,
              'note': 'The private 0.8 target cannot be verified or guaranteed before upload.'}
    u.atomic_json(ROOT / 'report.json', report)
    u.snapshot()
    log('Submission', SUBMISSION, 'Recovery', ARTIFACT, 'Changed labels', changes)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('action', choices=('prepare', 'train', 'infer', 'gate',
                                           'choose', 'package', 'snapshot'))
    parser.add_argument('--stage', default='pilot_joint')
    parser.add_argument('--split', default='test', choices=('dev', 'test'))
    parser.add_argument('--deadline', type=float, default=time.time() + MAX_HOURS * 3600)
    args = parser.parse_args()
    if args.action == 'prepare':
        prepare()
    elif args.action == 'train':
        run_stage(args.stage, args.deadline)
    elif args.action == 'infer':
        infer_stage(args.stage, args.split)
    elif args.action == 'gate':
        gate()
    elif args.action == 'choose':
        choose()
    elif args.action == 'package':
        package()
    elif args.action == 'snapshot':
        u.snapshot()


if __name__ == '__main__':
    main()




## Full-document section training and submission code

In [ ]:
%%writefile /kaggle/working/cs781_phase3_section_span_runner.py
"""Phase 3 full-document section and source tracker.

Embedded verbatim in the Run All notebook. All learned language-model weights
originate in the user's Phase 1 checkpoint 03. The old slot 06 is audited only;
it is never substituted for this notebook's submission.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from datetime import timedelta
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
from torch import nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.checkpoint import checkpoint

import cs781_phase3_hierarchical_runner as u
import cs781_phase3_context_joint_runner as c

WORK = Path('/kaggle/working')
ROOT = WORK / 'CS781_Capstone/phase3_sections'
ARTIFACT = WORK / 'CS781_Phase3_Sections_All_Artifacts.zip'
SUBMISSION = WORK / 'CS781_Phase3_Sections_Submission.zip'
u.ROOT = ROOT
u.ARTIFACT = ARTIFACT
u.OLD_ARCHIVES = ('CS781_Phase3_All_Artifacts.zip',
                  'CS781_Phase3_Context_Joint_All_Artifacts.zip',
                  'CS781_Phase3_Hierarchical_All_Artifacts.zip')


def required_source(name):
    return name=='runner.py' or name.startswith('official/') or name.startswith('bases/03/')


def source_ready():
    needed=('runner.py','official/CL_train.json','official/IT_train.json',
            'official/CL_dev.json','official/IT_dev.json','official/CL_test.json',
            'official/IT_test.json','official/label_vocab.json',
            'bases/03/model/prefix_hybrid_lm.py','bases/03/model/course_bpe.py',
            'bases/03/checkpoints/selected_model.pt','bases/03/tokenizer/tokenizer.json',
            'bases/03/tokenizer/tokenizer_metadata.json')
    return all((u.PRIOR/name).is_file() for name in needed)


# Use the older runner solely as an archive/data utility. Its slot-06 fallback
# is never called and is not required to reconstruct this experiment.
u.required_prior=required_source
u.prior_ready=source_ready

SEED = 781362
FOLDS = 5
LOCAL_MAX_EPOCHS = 7
LOCAL_INNER_EPOCHS = 5
GLOBAL_EPOCHS = 35
TOKEN_BUDGET = 8192
MAX_SENT_BATCH = 24
ACCUM = 2
MAX_SPAN = 64
ARCHIVE_SECONDS = 35 * 60
START_TIME_FILE = ROOT / 'run_start.json'
FAILED_RUN_SOURCE_SHA256 = 'd3653fac38eefb223f07a8deb5e1b0fb835afefaa6ff519b72305dea21da9c05'
SNAPSHOT_HANG_SOURCE_SHA256 = '14dfb977c41041eb4d9cf39056c8cb7b0dd90116f0b8c77fb602f3b9b44e1361'


def log(*parts):
    print(time.strftime('%Y-%m-%d %H:%M:%S'), *parts, flush=True)


def sample_gpus(stage,epoch):
    try:
        query=subprocess.run(['nvidia-smi','--query-gpu=index,utilization.gpu,memory.used,memory.total,power.draw',
                              '--format=csv,noheader,nounits'],capture_output=True,text=True,
                             check=True,timeout=15)
        values=[[v.strip() for v in row.split(',')] for row in query.stdout.strip().splitlines()]
        gpu=[{'index':int(v[0]),'utilization_percent':int(v[1]),'memory_mib':int(v[2]),
              'total_mib':int(v[3]),'power_watts':float(v[4])} for v in values]
    except Exception as exc:gpu=[{'error':repr(exc)}]
    with (ROOT/'gpu_samples.jsonl').open('a',encoding='utf8') as out:
        out.write(json.dumps({'time':time.time(),'stage':stage,'epoch':epoch,'gpus':gpu})+'\n')


def random_seed(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32-1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def pairs(docs):
    return {d['id']: d for d in docs}


def balanced_assignment(docs):
    groups=[[],[]];loads=[0,0]
    for doc in sorted(docs,key=lambda d:len(d['text']),reverse=True):
        rank=min(range(2),key=lambda i:loads[i])
        groups[rank].append(doc)
        loads[rank]+=len(doc['text'])
    return groups


def inner_split(docs, seed):
    mapping = {}
    for domain in ('CL', 'IT'):
        rows = [d for d in docs if d['domain'] == domain]
        rows.sort(key=lambda d: hashlib.sha256(f'{seed}:{d["id"]}'.encode()).hexdigest())
        for i, doc in enumerate(rows):
            mapping[doc['id']] = i % 2
    return mapping


FLAG_NAMES = (
    'heading', 'numbered_heading', 'argument_heading', 'analysis_heading',
    'fact_heading', 'dissent_heading', 'petitioner_word', 'respondent_word',
    'counsel', 'appeared_for', 'court_voice', 'lower_court', 'commission',
    'citation', 'quotation', 'statute', 'precedent', 'negative_precedent',
    'positive_precedent', 'overruled', 'submitted', 'held', 'therefore',
    'sentence_length', 'document_position', 'near_start', 'near_end', 'domain_it',
)


def lexical_flags(doc):
    rows = []
    n = len(doc['text'])
    for i, sentence in enumerate(doc['text']):
        t = sentence.lower().strip()
        short = len(t.split()) <= 12
        heading = short and (t.endswith(':') or sentence.strip().isupper() or
                             bool(re.search(r'\b(analysis|conclusion|arguments|submissions|facts|issues|order|dissent)\b', t)))
        numbered = bool(re.match(r'^\s*(?:\d+(?:\.\d+)*|[ivxlcdm]+)[.)\s]', t))
        values = [
            heading, numbered, short and bool(re.search(r'argument|submission', t)),
            short and bool(re.search(r'analysis|conclusion|reasoning', t)),
            short and bool(re.search(r'fact|background|history', t)),
            bool(re.search(r'\bdissent(?:ing)?\b', t)),
            bool(re.search(r'petitioners?|appellants?', t)),
            bool(re.search(r'respondents?|opposite part(?:y|ies)', t)),
            bool(re.search(r'counsel|advocate|solicitor', t)),
            'appeared for' in t or 'on behalf of' in t,
            bool(re.search(r'\bwe\b|\bour view\b|this court|we therefore', t)),
            bool(re.search(r'lower court|trial court|tribunal|below court', t)),
            bool(re.search(r'commission|\bcci\b|\bdg\b', t)),
            bool(re.search(r'\b\d{4}\b.*(?:scc|scr|air|plr|itr)|\bsupra\b|\bv\.\s', t)),
            '"' in sentence or "'" in sentence or 'extracted below' in t,
            bool(re.search(r'\bsection\b|\bact\b|\bstatute\b|\brule\b', t)),
            bool(re.search(r'precedent|judg(?:e)?ment|decision|case cited', t)),
            bool(re.search(r'not binding|distinguish(?:ed|able)|not applicable|inapplicable|no assistance|not relied|cannot be relied|rejected', t)),
            bool(re.search(r'relied upon|binding precedent|followed|squarely applies', t)),
            bool(re.search(r'overrul(?:ed|ing)', t)),
            bool(re.search(r'submitted|argued|contended|urged|pleaded', t)),
            bool(re.search(r'held|observed|concluded|ruled|ordered', t)),
            bool(re.search(r'therefore|accordingly|hence|for these reasons', t)),
            min(1., len(sentence.split()) / 100), i / max(1, n-1),
            i < 5, i >= n-5, doc['domain']=='IT',
        ]
        rows.append([float(v) for v in values])
    return torch.tensor(rows, dtype=torch.float32)


def prepare():
    if torch.cuda.device_count()!=2:
        raise RuntimeError('Enable exactly two Kaggle GPUs before Run All')
    ROOT.mkdir(parents=True, exist_ok=True)
    u.restore()
    old, train, dev, test, vocab = u.sets()
    if len(train) != 80 or len(dev) != 10 or len(test) != 10 or len(vocab) != 13:
        raise RuntimeError('Unexpected official Phase 3 data')
    conf = {'version': 1, 'seed': SEED, 'folds': u.fold_map(train),
            'local_max_epochs': LOCAL_MAX_EPOCHS, 'local_inner_epochs': LOCAL_INNER_EPOCHS,
            'max_span': MAX_SPAN, 'source_sha256': u.digest(__file__)}
    path = ROOT / 'config.json'
    if path.exists():
        previous=u.read_json(path)
        if previous!=conf:
            earlier=previous.get('source_sha256')
            expected=dict(conf,source_sha256=earlier)
            saved_source=ROOT/'section_span_runner.py'
            if (earlier not in {FAILED_RUN_SOURCE_SHA256,SNAPSHOT_HANG_SOURCE_SHA256}
                    or previous!=expected or not saved_source.is_file()
                    or u.digest(saved_source)!=earlier):
                raise RuntimeError('Recovery code/configuration mismatch')
            u.atomic_json(ROOT/'migration.json',{
                'from_source_sha256':earlier,
                'to_source_sha256':conf['source_sha256'],
                'reason':'Repair two-GPU auxiliary losses and move recovery snapshots outside DDP',
                'local_checkpoints_reused':True,'features_reused':True,
                'migrated_unix':time.time()})
            u.atomic_json(path,conf)
            log('Resuming saved models and features after runner repair')
    else:
        u.atomic_json(path, conf)
    shutil.copy2(Path(__file__), ROOT / 'section_span_runner.py')
    shutil.copy2(Path(u.__file__), ROOT / 'recovery_utilities.py')
    shutil.copy2(Path(c.__file__), ROOT / 'context_tokenizer.py')
    # Completed stages need only their selected weights, not large optimizer
    # states. Keep latest.pt for every unfinished stage so an interrupted
    # training stage can resume from its last epoch.
    for family in ('local','global'):
        for done in (ROOT/family).glob('*/done.json'):
            (done.parent/'latest.pt').unlink(missing_ok=True)
    if not START_TIME_FILE.exists():
        u.atomic_json(START_TIME_FILE, {'started_unix': time.time()})

    token_path = ROOT / 'tokens.pt'
    if not token_path.exists():
        live = WORK / 'CS781_Capstone/phase3_joint/tokens.pt'
        if live.is_file():
            shutil.copy2(live, token_path)
        else:
            prior_zip = u.locate('CS781_Phase3_Context_Joint_All_Artifacts.zip')
            if prior_zip:
                u.extract_manifest_zip(prior_zip, 'hier/', ROOT, lambda n: n == 'tokens.pt')
    if not token_path.exists():
        _, tokenizer = old.load_base('03')
        u.atomic_torch(token_path, c.make_tokens(train + dev + test, tokenizer))

    flags_path = ROOT / 'lexical_flags.pt'
    if not flags_path.exists():
        flags = {d['id']: lexical_flags(d) for d in train + dev + test}
        u.atomic_torch(flags_path, flags)
    runs = Counter()
    for d in train:
        y = d['labels']
        for i in range(len(y)):
            if i == 0 or y[i] != y[i-1]:
                runs[y[i]] += 1
    u.atomic_json(ROOT / 'audit.json', {
        'training_documents': len(train), 'development_documents': len(dev),
        'test_documents': len(test),
        'train_sentences': sum(len(d['text']) for d in train),
        'dev_sentences': sum(len(d['text']) for d in dev),
        'test_sentences': sum(len(d['text']) for d in test),
        'gold_segments_by_role': dict(runs),
        'feature_names': FLAG_NAMES,
        'source': 'Only official Phase 3 JSON and user Phase 1 base 03; no external weights.',
    })
    if not ARTIFACT.exists():
        u.snapshot()
    log('Prepared', ROOT)


def local_stage_docs(stage, train, config):
    if stage == 'full':
        return train, []
    if stage.startswith('final_inner_'):
        side = int(stage[-1])
        imap = inner_split(train, SEED + 900)
        return ([d for d in train if imap[d['id']] != side],
                [d for d in train if imap[d['id']] == side])
    kind, f, *tail = stage.split('_')
    fold = int(f)
    fitting = [d for d in train if config['folds'][d['id']] != fold]
    outer = [d for d in train if config['folds'][d['id']] == fold]
    if kind == 'outer':
        return fitting, outer
    if kind == 'inner':
        side = int(tail[0])
        imap = inner_split(fitting, SEED + fold)
        return ([d for d in fitting if imap[d['id']] != side],
                [d for d in fitting if imap[d['id']] == side])
    raise ValueError(stage)


class LocalEncoder(nn.Module):
    def __init__(self, base, classes=13):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        self.unfreeze_from = max(0, len(base.blocks)-8)
        for block in base.blocks[self.unfreeze_from:]:
            for p in block.parameters():p.requires_grad_(True)
        for p in base.final_norm.parameters():p.requires_grad_(True)
        width = base.hidden_size
        self.attention = nn.Linear(width, 1)
        self.project = nn.Sequential(nn.Linear(width*3, 512), nn.LayerNorm(512),
                                     nn.GELU(), nn.Dropout(.30))
        self.position = nn.Embedding(32, 512)
        self.domain = nn.Embedding(2, 512)
        self.classifier = nn.Sequential(nn.LayerNorm(512), nn.Dropout(.30),
                                        nn.Linear(512, classes))

    def forward(self, ids, mask, target, pos, domain):
        base = self.base
        x = base.embedding_dropout(base.token_embedding(ids) * math.sqrt(base.hidden_size))
        visible = base._prefix_hybrid_visibility(mask, mask.sum(1))
        cosine, sine = base.rotary(ids.size(1), x.dtype)
        fixed=x
        for i, block in enumerate(base.blocks):
            if i < self.unfreeze_from:
                with torch.no_grad():
                    x = block(x, visible, cosine, sine)
                fixed=x
            else:
                x = checkpoint(block, x, visible, cosine, sine, use_reentrant=False) if self.training else block(x, visible, cosine, sine)
        x = base.final_norm(x)
        a = self.attention(x).squeeze(-1).masked_fill(~target, -1e4).softmax(-1)
        pooled = (x*a.unsqueeze(-1)).sum(1)
        mean = (x*target.unsqueeze(-1)).sum(1)/target.sum(1,keepdim=True).clamp_min(1)
        maximum = x.masked_fill(~target.unsqueeze(-1),-1e4).max(1).values
        h = self.project(torch.cat((pooled,mean,maximum),-1))
        h = h + self.position(pos) + self.domain(domain)
        # The learned projection differs across cross-fit encoders. Shared
        # Phase 1 hidden coordinates keep document-head features aligned.
        stable=(fixed*target.unsqueeze(-1)).sum(1)/target.sum(1,keepdim=True).clamp_min(1)
        feature=torch.cat((stable.detach(),mean),-1)
        return feature, self.classifier(h)


def trainable_state(model):
    names = {name for name,p in model.named_parameters() if p.requires_grad}
    return {key:value.detach().cpu().clone() for key,value in model.state_dict().items()
            if key in names or key.startswith('project.') or key.startswith('classifier.')
            or key.startswith('attention.') or key.startswith('position.') or key.startswith('domain.')}


def load_trainable(model, state):
    unexpected = set(state) - set(model.state_dict())
    if unexpected:
        raise RuntimeError(f'Unknown adapter keys: {unexpected}')
    model.load_state_dict(state, strict=False)


def sample_rows(docs, epoch):
    rng = random.Random(SEED + epoch*1009)
    rows=[]
    for d in docs:
        n=len(d['text'])
        budget=min(n,max(48,round(6*math.sqrt(n))))
        anchors=[i for i,y in enumerate(d['labels']) if i==0 or y!=d['labels'][i-1]]
        rng.shuffle(anchors)
        chosen=set(anchors[:min(len(anchors),budget//3)])
        remaining=list(set(range(n))-chosen)
        rng.shuffle(remaining)
        chosen.update(remaining[:max(0,budget-len(chosen))])
        rows.extend((d['id'],i) for i in sorted(chosen))
    rng.shuffle(rows)
    return rows


def sentence_batches(rows, tokens, epoch):
    rng=random.Random(SEED+epoch*2039)
    pool=list(rows)
    rng.shuffle(pool)
    batches=[]
    for start in range(0,len(pool),384):
        group=pool[start:start+384]
        group.sort(key=lambda r:len(tokens[r[0]][r[1]][0]))
        j=0
        while j<len(group):
            width=len(tokens[group[j][0]][group[j][1]][0])
            size=min(MAX_SENT_BATCH,max(1,TOKEN_BUDGET//max(1,width)))
            batch=group[j:j+size]
            batches.append(batch)
            j+=size
    rng.shuffle(batches)
    while len(batches)%(2*ACCUM):
        batches.append(batches[len(batches)%max(1,len(batches))])
    return batches


def collate_sentences(rows, docs_by_id, tokens, pad, vocab, device):
    label_to_id={label:i for i,label in enumerate(vocab)}
    seqs=[tokens[did][i] for did,i in rows]
    width=max(len(seq[0]) for seq in seqs)
    n=len(rows)
    ids=torch.full((n,width),pad,dtype=torch.long)
    mask=torch.zeros((n,width),dtype=torch.bool)
    target=torch.zeros((n,width),dtype=torch.bool)
    pos=torch.zeros(n,dtype=torch.long)
    domain=torch.zeros(n,dtype=torch.long)
    labels=torch.full((n,),-1,dtype=torch.long)
    for j,((did,i),(seq,a,b)) in enumerate(zip(rows,seqs)):
        d=docs_by_id[did]
        ids[j,:len(seq)]=torch.tensor(seq,dtype=torch.long)
        mask[j,:len(seq)]=True
        target[j,a:b]=True
        pos[j]=min(31,int(32*i/max(1,len(d['text']))))
        domain[j]=0 if d['domain']=='CL' else 1
        if d['labels'] is not None:
            labels[j]=label_to_id[d['labels'][i]]
    def move(x):return x.pin_memory().to(device,non_blocking=True) if device.type=='cuda' else x
    return tuple(move(x) for x in (ids,mask,target,pos,domain,labels))


def local_optimizer(model, total_steps):
    groups={}
    for name,p in model.named_parameters():
        if not p.requires_grad:continue
        if name.startswith('base.blocks.'):
            layer=int(name.split('.')[2]);lr=6e-6+(1.8e-5-6e-6)*(layer-model.unfreeze_from)/max(1,len(model.base.blocks)-1-model.unfreeze_from)
        elif name.startswith('base.'):
            lr=1.8e-5
        else:lr=1.5e-4
        decay=0. if name.endswith('bias') or 'norm' in name.lower() else .03
        groups.setdefault((lr,decay),[]).append(p)
    try:opt=torch.optim.AdamW([{'params':ps,'lr':lr,'weight_decay':wd} for (lr,wd),ps in groups.items()],fused=True)
    except (TypeError,RuntimeError):opt=torch.optim.AdamW([{'params':ps,'lr':lr,'weight_decay':wd} for (lr,wd),ps in groups.items()])
    warm=max(8,int(.08*total_steps))
    def factor(step):
        if step<warm:return max(.1,(step+1)/warm)
        return .1+.9*.5*(1+math.cos(math.pi*min(1.,(step-warm)/max(1,total_steps-warm))))
    return opt,torch.optim.lr_scheduler.LambdaLR(opt,factor)


def local_model(old, device, vocab):
    base, tokenizer = old.load_base('03')
    model = LocalEncoder(base, len(vocab)).to(device)
    return model, tokenizer


@torch.no_grad()
def local_evaluate(model, docs, tokens, pad, vocab, rank, device):
    model.eval()
    own = {}
    for d in balanced_assignment(docs)[rank]:
        logits=[]
        for start in range(0,len(d['text']),MAX_SENT_BATCH):
            rows=[(d['id'],i) for i in range(start,min(len(d['text']),start+MAX_SENT_BATCH))]
            ids,mask,target,pos,domain,_=collate_sentences(rows,pairs([d]),tokens,pad,vocab,device)
            with torch.autocast('cuda',dtype=torch.float16):
                _,score=model(ids,mask,target,pos,domain)
            logits.append(score.float().cpu())
        own[d['id']]=torch.cat(logits).softmax(-1)
    result=u.gather_dict(own)
    if rank==0:
        return u.metrics(result,docs,vocab)
    return None


def train_local(stage):
    rank,device=u.init_dist()
    try:
        random_seed(SEED+sum(map(ord,stage))+rank)
        old,train,dev,_,vocab=u.sets()
        conf=u.read_json(ROOT/'config.json')
        fit,held=local_stage_docs(stage,train,conf)
        is_outer=stage.startswith('outer_')
        epochs=LOCAL_MAX_EPOCHS if is_outer or stage=='full' else LOCAL_INNER_EPOCHS
        validation=dev if is_outer or stage=='full' else []
        stage_dir=ROOT/'local'/stage
        stage_dir.mkdir(parents=True,exist_ok=True)
        if (stage_dir/'done.json').exists():
            if rank==0:
                latest=(stage_dir/'latest.pt')
                latest.unlink(missing_ok=True)
                log('Local stage already complete',stage)
            dist.barrier();return
        tokens=torch.load(ROOT/'tokens.pt',map_location='cpu',weights_only=False)
        model,tok=local_model(old,device,vocab)
        pad=int(tok.pad_token_id)
        count=len(sentence_batches(sample_rows(fit,1),tokens,1))//2//ACCUM
        opt,scheduler=local_optimizer(model,max(1,count*epochs))
        scaler=torch.amp.GradScaler('cuda')
        latest=stage_dir/'latest.pt'
        start=1;best=-1.;best_epoch=0
        if latest.exists():
            state=torch.load(latest,map_location='cpu',weights_only=False)
            load_trainable(model,state['model']);opt.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
            start=state['epoch']+1;best=state['best'];best_epoch=state['best_epoch']
        ddp=DDP(model,device_ids=[device.index],broadcast_buffers=False,
                find_unused_parameters=False,static_graph=True)
        class_w=u.class_weights(fit,vocab,device)
        docs_by_id=pairs(fit)
        history=u.read_json(stage_dir/'history.json') if (stage_dir/'history.json').exists() else []
        for epoch in range(start,epochs+1):
            ddp.train();opt.zero_grad(set_to_none=True)
            batches=sentence_batches(sample_rows(fit,epoch),tokens,epoch)
            rank_batches=batches[rank::2]
            total_loss=0.
            for j,rows in enumerate(rank_batches):
                ids,mask,target,pos,domain,y=collate_sentences(rows,docs_by_id,tokens,pad,vocab,device)
                with torch.autocast('cuda',dtype=torch.float16):
                    _,score=ddp(ids,mask,target,pos,domain)
                    loss=F.cross_entropy(score.float(),y,weight=class_w,label_smoothing=.06)/ACCUM
                u.finite_or_fail(loss)
                scaler.scale(loss).backward()
                total_loss+=float(loss.detach())*ACCUM
                if (j+1)%ACCUM==0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                    scaler.step(opt);scaler.update();scheduler.step()
                    opt.zero_grad(set_to_none=True)
            dist.barrier()
            evaluated=local_evaluate(ddp.module,validation,tokens,pad,vocab,rank,device) if validation else None
            if rank==0:
                metric=evaluated['macro_supported'] if evaluated else -total_loss/max(1,len(rank_batches))
                take=(metric>best) if validation else True
                if take:
                    best=metric;best_epoch=epoch
                    u.atomic_torch(stage_dir/'best.pt',{'model':trainable_state(model),'epoch':epoch,
                        'stage':stage,'fit_ids':[d['id'] for d in fit],'score':metric})
                history.append({'epoch':epoch,'train_loss_rank0':total_loss/max(1,len(rank_batches)),
                                'development':evaluated,'best_epoch':best_epoch})
                u.atomic_json(stage_dir/'history.json',history)
                u.atomic_torch(latest,{'model':trainable_state(model),'optimizer':opt.state_dict(),
                    'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
                    'epoch':epoch,'best':best,'best_epoch':best_epoch})
                log('Local',stage,'epoch',epoch,'loss',round(history[-1]['train_loss_rank0'],4),
                    'dev F1',None if not evaluated else round(metric,4))
                sample_gpus('local_'+stage,epoch)
            dist.barrier()
        if rank==0:
            u.atomic_json(stage_dir/'done.json',{'stage':stage,'selected_epoch':best_epoch,
                'fit_ids':[d['id'] for d in fit],'extraction_ids':[d['id'] for d in held],
                'validation':history[best_epoch-1]['development'] if validation else None})
            latest.unlink(missing_ok=True)
        dist.barrier()
    finally:
        dist.destroy_process_group()


def extraction_docs(stage,train,dev,test,conf):
    _,held=local_stage_docs(stage,train,conf)
    if stage=='full':return dev+test
    if stage.startswith('outer_'):return held+dev
    return held


def extract_local(stage):
    rank,device=u.init_dist()
    try:
        old,train,dev,test,vocab=u.sets()
        conf=u.read_json(ROOT/'config.json')
        docs=extraction_docs(stage,train,dev,test,conf)
        stage_dir=ROOT/'local'/stage
        if not (stage_dir/'done.json').is_file():raise RuntimeError('Train local stage first: '+stage)
        feature_dir=ROOT/'features'/stage
        feature_dir.mkdir(parents=True,exist_ok=True)
        tokens=torch.load(ROOT/'tokens.pt',map_location='cpu',weights_only=False)
        model,tok=local_model(old,device,vocab)
        state=torch.load(stage_dir/'best.pt',map_location='cpu',weights_only=False)
        load_trainable(model,state['model']);model.eval()
        pad=int(tok.pad_token_id)
        for doc in balanced_assignment(docs)[rank]:
            destination=feature_dir/(doc['id']+'.pt')
            if destination.is_file():continue
            embeddings=[];logits=[]
            with torch.no_grad():
                for start in range(0,len(doc['text']),MAX_SENT_BATCH):
                    rows=[(doc['id'],i) for i in range(start,min(len(doc['text']),start+MAX_SENT_BATCH))]
                    ids,mask,target,pos,domain,_=collate_sentences(rows,pairs([doc]),tokens,pad,vocab,device)
                    with torch.autocast('cuda',dtype=torch.float16):
                        emb,score=model(ids,mask,target,pos,domain)
                    embeddings.append(emb.cpu().half());logits.append(score.cpu().half())
            data={'embedding':torch.cat(embeddings),'logits':torch.cat(logits),
                  'doc_id':doc['id'],'n':len(doc['text']),'stage':stage}
            u.atomic_torch(destination,data)
            log('Extracted',stage,doc['id'],len(doc['text']))
        dist.barrier()
        if rank==0:
            for doc in docs:
                p=feature_dir/(doc['id']+'.pt')
                if not p.is_file():raise RuntimeError('Missing features: '+str(p))
                data=torch.load(p,map_location='cpu',weights_only=False)
                if data['n']!=len(doc['text']) or data['embedding'].shape!=(len(doc['text']),768):
                    raise RuntimeError('Bad feature shape: '+doc['id'])
            u.atomic_json(feature_dir/'done.json',{'stage':stage,'docs':[d['id'] for d in docs]})
        dist.barrier()
    finally:dist.destroy_process_group()


def feature_path(stage, did):
    return ROOT/'features'/stage/(did+'.pt')


_STAGE_MAP={}


def head_feature_stage(mode, doc, conf):
    mode=str(mode)
    if mode not in _STAGE_MAP:
        all_train=u.sets()[1]
        if mode=='final':
            sides=inner_split(all_train,SEED+900)
            _STAGE_MAP[mode]={did:f'final_inner_{side}' for did,side in sides.items()}
            _STAGE_MAP[mode]['__other__']='full'
        else:
            fold=int(mode)
            fit=[d for d in all_train if conf['folds'][d['id']]!=fold]
            sides=inner_split(fit,SEED+fold)
            mapping={did:f'inner_{fold}_{side}' for did,side in sides.items()}
            mapping.update({d['id']:f'outer_{fold}' for d in all_train if conf['folds'][d['id']]==fold})
            mapping['__other__']=f'outer_{fold}'
            _STAGE_MAP[mode]=mapping
    return _STAGE_MAP[mode].get(doc['id'],_STAGE_MAP[mode]['__other__'])


def load_feature(stage,doc,device,flags):
    obj=torch.load(feature_path(stage,doc['id']),map_location='cpu',weights_only=False)
    if obj['n']!=len(doc['text']):raise RuntimeError('Feature count mismatch '+doc['id'])
    emb=obj['embedding'].float()
    logits=obj['logits'].float()
    lexical=flags[doc['id']]
    if lexical.shape!=(len(doc['text']),len(FLAG_NAMES)):
        raise RuntimeError('Lexical feature count mismatch '+doc['id'])
    values=torch.cat([emb,logits.softmax(-1),lexical],dim=-1)
    return values.pin_memory().to(device,non_blocking=True)


def gold_segments(labels,vocab):
    ids=[vocab.index(y) for y in labels]
    segments=[]
    start=0
    while start<len(ids):
        end=start+1
        while end<len(ids) and ids[end]==ids[start] and end-start<MAX_SPAN:end+=1
        segments.append((start,end,ids[start]))
        start=end
    return segments


class SectionTracker(nn.Module):
    def __init__(self,classes=13):
        super().__init__()
        self.input=nn.Sequential(nn.Linear(768+13+len(FLAG_NAMES),256),
                                 nn.LayerNorm(256),nn.GELU(),nn.Dropout(.30))
        self.conv=nn.ModuleList([nn.Conv1d(256,256,5,padding=2*d,dilation=d,groups=1)
                                 for d in (1,2,4)])
        self.conv_norm=nn.LayerNorm(256)
        self.gru=nn.GRU(256,128,num_layers=2,batch_first=True,bidirectional=True,dropout=.25)
        layer=nn.TransformerEncoderLayer(256,8,512,dropout=.20,activation='gelu',
                                          batch_first=True,norm_first=True)
        self.blocks=nn.TransformerEncoder(layer,2)
        self.final_gru=nn.GRU(256,128,batch_first=True,bidirectional=True)
        self.norm=nn.LayerNorm(256)
        self.role=nn.Linear(256,classes)
        self.boundary=nn.Sequential(nn.Linear(512,128),nn.GELU(),nn.Dropout(.25),nn.Linear(128,1))
        self.argument=nn.Linear(256,2)
        self.precedent=nn.Linear(256,2)
        self.span=nn.Sequential(nn.Linear(256*3+1,256),nn.LayerNorm(256),
                                nn.GELU(),nn.Dropout(.25),nn.Linear(256,classes))

    def span_logits(self,h,starts,ends):
        prefix=F.pad(h.cumsum(0),(0,0,1,0))
        mean=(prefix[ends]-prefix[starts])/(ends-starts).unsqueeze(-1)
        lengths=((ends-starts).float().log1p()/math.log1p(MAX_SPAN)).unsqueeze(-1)
        return self.span(torch.cat([h[starts],h[ends-1],mean,lengths],-1))

    def forward(self,features,span_starts=None,span_ends=None):
        x=self.input(features)
        for conv in self.conv:
            x=self.conv_norm(x+F.gelu(conv(x.T.unsqueeze(0)).squeeze(0).T))
        recurrent,_=self.gru(x.unsqueeze(0))
        h=x+recurrent[0]
        summaries=[]
        for start in range(0,len(h),64):summaries.append(h[start:start+64].mean(0))
        global_state=self.blocks(torch.stack(summaries)[None])[0]
        global_state=torch.repeat_interleave(global_state,64,dim=0)[:len(h)]
        h=self.norm(h+global_state)
        contextual,_=self.final_gru(h[None])
        h=self.norm(h+contextual[0])
        boundary=self.boundary(torch.cat([h[:-1],h[1:]],-1)).squeeze(-1)
        spans=self.span_logits(h,span_starts,span_ends) if span_starts is not None else None
        return h,self.role(h),boundary,self.argument(h),self.precedent(h),spans


def global_weights(docs,vocab,device):
    counts=Counter(y for d in docs for y in d['labels'])
    typical=sum(counts.values())/len(vocab)
    values=[min(3.5,max(.65,math.sqrt(typical/max(1,counts[y])))) for y in vocab]
    result=torch.tensor(values,device=device,dtype=torch.float32)
    return result/result.mean()


def global_loss(output,doc,vocab,weights,device):
    h,role,boundary,argument,precedent,span=output
    y=torch.tensor([vocab.index(z) for z in doc['labels']],device=device)
    role_loss=F.cross_entropy(role.float(),y,weight=weights,label_smoothing=.04)
    edge=(y[:-1]!=y[1:]).float()
    edge_loss=F.binary_cross_entropy_with_logits(boundary.float(),edge,
                pos_weight=torch.tensor(3.,device=device)) if len(edge) else role_loss*0
    starts,ends,roles=zip(*gold_segments(doc['labels'],vocab))
    span_y=torch.tensor(roles,device=device)
    span_loss=F.cross_entropy(span.float(),span_y,weight=weights,label_smoothing=.02)
    # Every document must touch both auxiliary heads. Some documents contain
    # neither pair of labels; a zero-valued term keeps DDP's gradient set stable.
    pair_loss=argument.float().sum()*0+precedent.float().sum()*0
    for names,logits in ((('ArgumentPetitioner','ArgumentRespondent'),argument),
                          (('PrecedentReliedUpon','PrecedentNotReliedUpon'),precedent)):
        take=(y==vocab.index(names[0]))|(y==vocab.index(names[1]))
        if bool(take.any()):
            pair_target=(y[take]==vocab.index(names[1])).long()
            pair_loss=pair_loss+F.cross_entropy(logits[take].float(),pair_target)
    return role_loss+.40*edge_loss+.40*span_loss+.12*pair_loss


def transition_prior(docs,vocab):
    counts=np.ones((len(vocab),len(vocab)),dtype=np.float64)*.5
    for d in docs:
        seg=gold_segments(d['labels'],vocab)
        for (_,_,a),(_,_,b) in zip(seg,seg[1:]):counts[a,b]+=1
    counts=counts/counts.sum(1,keepdims=True)
    return np.log(counts).astype('float32')


@torch.no_grad()
def decode_sections(model,h,role,boundary,prior,role_override=None):
    """Duration-limited semi-Markov decoder; same-label adjacent spans are valid."""
    scores=role_override if role_override is not None else role
    n,classes=scores.shape
    token_log=F.log_softmax(scores.float(),-1)
    prefix=F.pad(token_log.cumsum(0),(0,0,1,0))
    edge_yes=F.logsigmoid(boundary.float())
    edge_no=F.logsigmoid(-boundary.float())
    edge_prefix=F.pad(edge_no.cumsum(0),(1,0))
    candidates=np.full((n+1,MAX_SPAN,classes),-1e9,dtype='float32')
    for length in range(1,min(MAX_SPAN,n)+1):
        starts=torch.arange(n-length+1,device=h.device)
        ends=starts+length
        role_part=.70*(prefix[ends]-prefix[starts])
        interior=edge_prefix[ends-1]-edge_prefix[starts]
        begins=torch.zeros_like(interior)
        active=starts>0
        begins[active]=edge_yes[starts[active]-1]
        boundary_part=.4*(interior+begins)
        span_parts=[]
        for a in range(0,len(starts),8192):
            span_parts.append(F.log_softmax(model.span_logits(h,starts[a:a+8192],ends[a:a+8192]).float(),-1))
        span_part=torch.cat(span_parts)
        joint=role_part+.85*span_part+boundary_part[:,None]
        candidates[ends.cpu().numpy(),length-1,:]=joint.cpu().numpy()
    transitions=.16*prior.copy()
    np.fill_diagonal(transitions,0.)
    dp=np.full((n+1,classes),-1e20,dtype='float32')
    back_start=np.zeros((n+1,classes),dtype='int32')
    back_label=np.zeros((n+1,classes),dtype='int16')
    for end in range(1,n+1):
        lengths=np.arange(1,min(MAX_SPAN,end)+1)
        starts=end-lengths
        previous=dp[starts,:,None]+transitions[None,:,:]
        prev_label=previous.argmax(axis=1)
        previous_best=previous.max(axis=1)
        previous_best[starts==0]=0.
        prev_label[starts==0]=0
        totals=previous_best+candidates[end,lengths-1,:]
        best_len=totals.argmax(axis=0)
        dp[end]=totals[best_len,np.arange(classes)]
        back_start[end]=starts[best_len]
        back_label[end]=prev_label[best_len,np.arange(classes)]
    labels=np.zeros(n,dtype='int16')
    end=n;role_id=int(dp[end].argmax())
    while end>0:
        start=int(back_start[end,role_id])
        labels[start:end]=role_id
        prev=int(back_label[end,role_id]);end=start;role_id=prev
    return labels


@torch.no_grad()
def evaluate_global(model,docs,mode,conf,flags,vocab,rank,device,structured=False,prior=None):
    model.eval();own={}
    for d in balanced_assignment(docs)[rank]:
        stage=head_feature_stage(mode,d,conf)
        x=load_feature(stage,d,device,flags)
        h,role,boundary,_,_,_=model(x)
        local=torch.load(feature_path(stage,d['id']),map_location='cpu',weights_only=False)['logits'].float()
        entry={'local':local.softmax(-1),'global':role.float().softmax(-1).cpu(),
               'blend':(.35*local.softmax(-1)+.65*role.float().cpu().softmax(-1))}
        if structured:
            entry['section']=torch.from_numpy(decode_sections(model,h,role,boundary,prior))
            blend_log=torch.log(entry['blend'].to(device).clamp_min(1e-7))
            entry['blend_section']=torch.from_numpy(decode_sections(model,h,role,boundary,prior,blend_log))
        own[d['id']]=entry
    return u.gather_dict(own)


def result_metrics(results,docs,vocab):
    names=results[docs[0]['id']].keys()
    output={}
    for name in names:
        if name.endswith('section'):
            probs={did:F.one_hot(rows[name].long(),len(vocab)).float() for did,rows in results.items()}
        else:probs={did:rows[name] for did,rows in results.items()}
        output[name]=u.metrics(probs,docs,vocab)
    return output


def save_global_results(path,results,docs,vocab):
    u.atomic_torch(path,results)
    metrics=result_metrics(results,docs,vocab)
    u.atomic_json(path.with_suffix('.json'),metrics)
    return metrics


def global_documents(mode,train,dev,conf):
    if mode=='final':return train,dev,[]
    fold=int(mode)
    return ([d for d in train if conf['folds'][d['id']]!=fold],dev,
            [d for d in train if conf['folds'][d['id']]==fold])


def train_global(mode):
    mode=str(mode)
    rank,device=u.init_dist()
    try:
        random_seed(SEED+421+rank+(100 if mode=='final' else int(mode)))
        _,train,dev,_,vocab=u.sets()
        conf=u.read_json(ROOT/'config.json')
        fitting,validation,outer=global_documents(mode,train,dev,conf)
        for doc in fitting+validation+outer:
            stage=head_feature_stage(mode,doc,conf)
            if not feature_path(stage,doc['id']).is_file():
                raise RuntimeError('Missing out-of-sample feature '+stage+'/'+doc['id'])
        flags=torch.load(ROOT/'lexical_flags.pt',map_location='cpu',weights_only=False)
        stage_dir=ROOT/'global'/mode
        stage_dir.mkdir(parents=True,exist_ok=True)
        if (stage_dir/'done.json').exists():
            if rank==0:
                (stage_dir/'latest.pt').unlink(missing_ok=True)
                log('Global stage already complete',mode)
            dist.barrier();return
        model=SectionTracker(len(vocab)).to(device)
        ddp=DDP(model,device_ids=[device.index],broadcast_buffers=False,
                find_unused_parameters=False)
        try:optimizer=torch.optim.AdamW(model.parameters(),lr=7e-4,weight_decay=.08,fused=True)
        except (TypeError,RuntimeError):optimizer=torch.optim.AdamW(model.parameters(),lr=7e-4,weight_decay=.08)
        steps=math.ceil(len(fitting)/2)*GLOBAL_EPOCHS
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,steps,eta_min=5e-5)
        scaler=torch.amp.GradScaler('cuda')
        weights=global_weights(fitting,vocab,device)
        latest=stage_dir/'latest.pt'
        start=1;best=-1.;best_epoch=0
        history=u.read_json(stage_dir/'history.json') if (stage_dir/'history.json').exists() else []
        if latest.exists():
            state=torch.load(latest,map_location='cpu',weights_only=False)
            model.load_state_dict(state['model'])
            optimizer.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler'])
            scaler.load_state_dict(state['scaler'])
            start=state['epoch']+1;best=state['best'];best_epoch=state['best_epoch']
        prior=transition_prior(fitting,vocab)
        if rank==0:u.atomic_torch(stage_dir/'transition.pt',prior)
        dist.barrier()
        for epoch in range(start,GLOBAL_EPOCHS+1):
            ddp.train()
            order=sorted(fitting,key=lambda d:len(d['text']))
            paired=[order[i:i+2] for i in range(0,len(order),2)]
            random.Random(SEED+epoch*41).shuffle(paired)
            order=[doc for pair in paired for doc in pair]
            if len(order)%2:order.append(order[-1])
            total=0.
            for doc in order[rank::2]:
                features=load_feature(head_feature_stage(mode,doc,conf),doc,device,flags)
                segments=gold_segments(doc['labels'],vocab)
                starts=torch.tensor([s for s,_,_ in segments],device=device,dtype=torch.long)
                ends=torch.tensor([e for _,e,_ in segments],device=device,dtype=torch.long)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast('cuda',dtype=torch.float16):
                    output=ddp(features,starts,ends)
                    loss=global_loss(output,doc,vocab,weights,device)
                u.finite_or_fail(loss)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                scaler.step(optimizer);scaler.update();scheduler.step()
                total+=float(loss.detach())
            dist.barrier()
            results=evaluate_global(model,validation,mode,conf,flags,vocab,rank,device)
            if rank==0:
                report=result_metrics(results,validation,vocab)
                metric=report['global']['macro_supported']
                if metric>best:
                    best=metric;best_epoch=epoch
                    u.atomic_torch(stage_dir/'best.pt',{'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                        'epoch':epoch,'dev':report,'mode':mode})
                history.append({'epoch':epoch,'train_loss_rank0':total/max(1,len(order)//2),
                                'development':report,'best_epoch':best_epoch})
                u.atomic_json(stage_dir/'history.json',history)
                u.atomic_torch(latest,{'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                    'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                    'scaler':scaler.state_dict(),'epoch':epoch,'best':best,'best_epoch':best_epoch})
                log('Global',mode,'epoch',epoch,'loss',round(history[-1]['train_loss_rank0'],4),
                    'dev F1',round(metric,4),'best',round(best,4))
                sample_gpus('global_'+mode,epoch)
                stop=epoch>=10 and epoch-best_epoch>=7
            else:stop=False
            signal=torch.tensor([int(stop)],device=device)
            dist.broadcast(signal,0)
            dist.barrier()
            if int(signal.item()):break
        state=torch.load(stage_dir/'best.pt',map_location='cpu',weights_only=False)
        model.load_state_dict(state['model']);model.eval()
        dev_results=evaluate_global(model,validation,mode,conf,flags,vocab,rank,device,True,prior)
        fold_results=evaluate_global(model,outer,mode,conf,flags,vocab,rank,device,True,prior) if outer else None
        if rank==0:
            dev_metrics=save_global_results(stage_dir/'development.pt',dev_results,validation,vocab)
            fold_metrics=save_global_results(stage_dir/'heldout.pt',fold_results,outer,vocab) if outer else None
            u.atomic_json(stage_dir/'done.json',{'mode':mode,'best_epoch':best_epoch,
                'best_dev_raw_supported_f1':best,'development':dev_metrics,'heldout':fold_metrics,
                'fitting_ids':[d['id'] for d in fitting],
                'heldout_ids':[d['id'] for d in outer]})
            latest.unlink(missing_ok=True)
        dist.barrier()
    finally:dist.destroy_process_group()


def infer_test():
    rank,device=u.init_dist()
    try:
        _,train,dev,test,vocab=u.sets()
        conf=u.read_json(ROOT/'config.json')
        stage_dir=ROOT/'global'/'final'
        if not (stage_dir/'done.json').exists():raise RuntimeError('Train final global tracker first')
        model=SectionTracker(len(vocab)).to(device)
        state=torch.load(stage_dir/'best.pt',map_location='cpu',weights_only=False)
        model.load_state_dict(state['model']);model.eval()
        prior=torch.load(stage_dir/'transition.pt',map_location='cpu',weights_only=False)
        flags=torch.load(ROOT/'lexical_flags.pt',map_location='cpu',weights_only=False)
        result=evaluate_global(model,test,'final',conf,flags,vocab,rank,device,True,prior)
        if rank==0:
            u.atomic_torch(ROOT/'test_results.pt',result)
            log('Inferred',len(result),'test documents; all results come from the new trained model')
        dist.barrier()
    finally:dist.destroy_process_group()


def select():
    _,train,dev,_,vocab=u.sets()
    names=('local','global','blend','section','blend_section')
    final_dir=ROOT/'global'/'final'
    if not (final_dir/'development.pt').exists():raise RuntimeError('Final model missing')
    final_dev=torch.load(final_dir/'development.pt',map_location='cpu',weights_only=False)
    final_metrics=result_metrics(final_dev,dev,vocab)
    done_folds=[f for f in range(FOLDS) if (ROOT/'global'/str(f)/'done.json').exists()]
    oof={};heldout=[]
    for f in done_folds:
        docs=[d for d in train if u.read_json(ROOT/'config.json')['folds'][d['id']]==f]
        result=torch.load(ROOT/'global'/str(f)/'heldout.pt',map_location='cpu',weights_only=False)
        if set(result)!={d['id'] for d in docs}:raise RuntimeError('Incomplete fold result '+str(f))
        overlap=set(oof)&set(result)
        if overlap:raise RuntimeError('Duplicate OOF documents '+str(overlap))
        oof.update(result);heldout.extend(docs)
    oof_metrics=result_metrics(oof,heldout,vocab) if heldout else None
    if len(done_folds)>=3:
        choice=max(names,key=lambda name:oof_metrics[name]['macro_supported'])
        basis='document-held-out pooled OOF macro F1'
    else:
        choice=max(names,key=lambda name:final_metrics[name]['macro_supported'])
        basis='official development macro F1; fewer than three outer folds completed'
    report={'selected':choice,'basis':basis,'completed_outer_folds':done_folds,
            'oof_documents':len(heldout),'oof':oof_metrics,'development':final_metrics,
            'private_score':None,'target_private_macro_f1':.8,
            'caution':'0.8 is an objective; a Kaggle evaluation is required to measure it.',
            'prior_submission_used_as_output':False}
    u.atomic_json(ROOT/'selection.json',report)
    log('Selected',choice,'based on',basis,'development F1',round(final_metrics[choice]['macro_supported'],4),
        'OOF F1',None if not oof_metrics else round(oof_metrics[choice]['macro_supported'],4))


def prior_six_predictions():
    source=u.locate('CS781_Phase3_Submission_06.zip')
    if source is None:return None
    with zipfile.ZipFile(source) as z:
        if 'prediction.json' not in z.namelist():return None
        return json.loads(z.read('prediction.json'))


def package():
    _,_,_,test,vocab=u.sets()
    if not (ROOT/'test_results.pt').is_file():raise RuntimeError('Inference missing')
    selection=u.read_json(ROOT/'selection.json')
    chosen=selection['selected']
    results=torch.load(ROOT/'test_results.pt',map_location='cpu',weights_only=False)
    if set(results)!={d['id'] for d in test}:raise RuntimeError('Test ID mismatch')
    predictions={}
    for doc in test:
        value=results[doc['id']][chosen]
        labels=value.tolist() if chosen.endswith('section') else value.argmax(-1).tolist()
        predictions[doc['id']]=[vocab[int(i)] for i in labels]
        if len(labels)!=len(doc['text']):raise RuntimeError('Test sentence count mismatch')
    if any(y not in vocab for rows in predictions.values() for y in rows):
        raise RuntimeError('Unknown prediction label')
    old=prior_six_predictions()
    changed=None
    if old is not None and set(old)==set(predictions):
        changed=sum(a!=b for did in predictions for a,b in zip(predictions[did],old[did]))
    if changed==0:
        log('WARNING: new model predictions equal the old slot 06; inspect before upload')
    u.atomic_json(ROOT/'prediction.json',predictions)
    temp=SUBMISSION.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ROOT/'prediction.json','prediction.json')
    os.replace(temp,SUBMISSION)
    with zipfile.ZipFile(SUBMISSION) as z:
        if z.namelist()!=['prediction.json'] or json.loads(z.read('prediction.json'))!=predictions:
            raise RuntimeError('Submission verification failed')
    shutil.copy2(SUBMISSION,WORK/'submission.zip')
    shutil.copy2(SUBMISSION,ROOT/'submission.zip')
    pilot_path=ROOT/'pilot_gate.json'
    pilot=u.read_json(pilot_path) if pilot_path.exists() else None
    status=('WEAK_HELDOUT_PILOT_REVIEW_BEFORE_UPLOAD' if pilot and not pilot['continue_outer_folds']
            else 'READY_FOR_MANUAL_UPLOAD')
    u.atomic_json(ROOT/'report.json',{'status':status,
        'submission':str(SUBMISSION),'submission_sha256':u.digest(SUBMISSION),
        'artifact':str(ARTIFACT),'selected':chosen,'selection_basis':selection['basis'],
        'completed_outer_folds':selection['completed_outer_folds'],
        'development_supported_macro_f1':selection['development'][chosen]['macro_supported'],
        'oof_supported_macro_f1':None if selection['oof'] is None else selection['oof'][chosen]['macro_supported'],
        'changed_labels_vs_slot06':changed,'test_documents':len(test),
        'test_sentences':sum(len(d['text']) for d in test),'private_score':None,
        'pilot_gate':pilot,
        'note':'A 0.8 private score is a target, not a measured or guaranteed result.'})
    u.snapshot()
    with zipfile.ZipFile(ARTIFACT) as z:
        integrity=json.loads(z.read('integrity.json'))['files']
        for name in ('hier/report.json','hier/prediction.json','hier/submission.zip',
                     'hier/test_results.pt','hier/global/final/best.pt','prior/bases/03/checkpoints/selected_model.pt'):
            if name not in integrity:raise RuntimeError('Recovery missing '+name)
    log('Submission',SUBMISSION,'artifact',ARTIFACT,'changed labels',changed)


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('action',choices=('prepare','local','extract','global','infer-test',
                                          'select','package','snapshot'))
    parser.add_argument('--stage',default=None)
    args=parser.parse_args()
    if args.action=='prepare':prepare()
    elif args.action=='local':train_local(args.stage)
    elif args.action=='extract':extract_local(args.stage)
    elif args.action=='global':train_global(args.stage)
    elif args.action=='infer-test':infer_test()
    elif args.action=='select':select()
    elif args.action=='package':package()
    elif args.action=='snapshot':u.snapshot()


if __name__=='__main__':main()



## Run the final model and the document-fold audit

In [ ]:
from pathlib import Path
import json, subprocess, sys, time

runner = Path('/kaggle/working/cs781_phase3_section_span_runner.py')
root = Path('/kaggle/working/CS781_Capstone/phase3_sections')
session_started = time.time()
TIME_BUDGET_HOURS = 9.5

def run(action, stage=None, distributed=False):
    command = [sys.executable]
    if distributed:
        command += ['-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2']
    command += [str(runner), action]
    if stage is not None:
        command += ['--stage', str(stage)]
    print('RUN', ' '.join(command), flush=True)
    subprocess.run(command, check=True)

def archive_after_stage():
    # Archive in its own process, after both GPU workers have exited. A timed
    # out archive leaves the previous verified ZIP intact for recovery.
    command = [sys.executable, str(runner), 'snapshot']
    print('ARCHIVE', ' '.join(command), flush=True)
    try:
        subprocess.run(command, check=True, timeout=900)
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print('Recovery refresh failed; previous ZIP remains intact:', repr(exc), flush=True)

run('prepare')

def ensure_local_features(stage):
    if (root / 'local' / stage / 'done.json').exists() and (root / 'features' / stage / 'done.json').exists():
        print('REUSE trained encoder and extracted features:', stage, flush=True)
        return
    run('local', stage, True)
    run('extract', stage, True)
    archive_after_stage()

def ensure_global(stage):
    if (root / 'global' / str(stage) / 'done.json').exists():
        print('REUSE trained section tracker:', stage, flush=True)
        return
    run('global', stage, True)
    archive_after_stage()

# A new model is fitted and packaged first. Completed stages are reused.
for stage in ('final_inner_0', 'final_inner_1', 'full'):
    ensure_local_features(stage)
ensure_global('final')
if not (root / 'test_results.pt').exists():
    run('infer-test', distributed=True)
else:
    print('REUSE saved test inference', flush=True)
run('select')
run('package')

# Add untouched document-fold evidence while the session budget permits.
for fold in range(5):
    elapsed = (time.time() - session_started) / 3600
    if elapsed > TIME_BUDGET_HOURS - 1.0:
        print('Stopping before fold', fold, 'to preserve packaging time.', flush=True)
        break
    for stage in (f'inner_{fold}_0', f'inner_{fold}_1', f'outer_{fold}'):
        ensure_local_features(stage)
    ensure_global(str(fold))
    run('select')
    if fold == 0:
        selection = json.loads(Path('/kaggle/working/CS781_Capstone/phase3_sections/selection.json').read_text())
        candidate_f1 = {name: row['macro_supported'] for name, row in selection['oof'].items()}
        best_pilot = max(candidate_f1.values())
        pilot_pass = best_pilot >= 0.50
        gate = {'best_fold0_supported_macro_f1': best_pilot, 'candidate_f1': candidate_f1,
                'continue_at_or_above': 0.50, 'continue_outer_folds': pilot_pass}
        Path('/kaggle/working/CS781_Capstone/phase3_sections/pilot_gate.json').write_text(
            json.dumps(gate, indent=2) + '\n')
        print('FOLD 0 PILOT', json.dumps(gate), flush=True)
    run('package')
    if fold == 0 and not pilot_pass:
        print('Stopping remaining folds: the section pilot did not show a meaningful held-out gain.', flush=True)
        break

report = json.loads(Path('/kaggle/working/CS781_Capstone/phase3_sections/report.json').read_text())
print(json.dumps(report, indent=2), flush=True)


## Verify the submission and portable recovery archive

In [ ]:
from pathlib import Path
import hashlib, json, zipfile

submission = Path('/kaggle/working/CS781_Phase3_Sections_Submission.zip')
recovery = Path('/kaggle/working/CS781_Phase3_Sections_All_Artifacts.zip')
with zipfile.ZipFile(submission) as z:
    assert z.namelist() == ['prediction.json']
    predictions = json.loads(z.read('prediction.json'))
assert len(predictions) == 10 and sum(map(len, predictions.values())) == 2197
with zipfile.ZipFile(recovery) as z:
    manifest = json.loads(z.read('integrity.json'))['files']
    assert set(manifest) == set(z.namelist()) - {'integrity.json'}
    for name, expected in manifest.items():
        digest = hashlib.sha256()
        size = 0
        with z.open(name) as member:
            for block in iter(lambda: member.read(4 << 20), b''):
                size += len(block)
                digest.update(block)
        assert size == expected['bytes'] and digest.hexdigest() == expected['sha256'], name
print('Submission and full recovery ZIP verified.')
print(json.dumps(json.loads(Path('/kaggle/working/CS781_Capstone/phase3_sections/report.json').read_text()), indent=2))
